<!-- artqr-doc -->
# ArtQR Fusion Notebook

Bu notebook, Google Colab icinde logo/artwork + okunabilir QR kod uretmek icin hazirlanmis ticari bir teslim pipeline'idir. Logo-aware halftone motoru logoyu sadece QR ustune koymaz; logo maskesini kontur, ic dolgu, acik detay ve dis halo katmanlarina ayirip QR modullerini bu katmanlara gore cizer.

## Fiverr / Upwork kullanim akisi

1. **Blok 3:** Musterinin logo veya artwork dosyasini yukle.
2. **Blok 4:** `service_tier`, `qr_data` ve kalite profilini sec. Premium islerde `service_tier = premium`, `quality_mode = en_yuksek_kalite` onerilir.
3. **Blok 5:** Musteri brief bilgilerini, `portfolio_example` veya `style_preset` degerini sec. Basit islerde teknik ayarlara dokunma.
4. **Blok 8-12:** Uretim, otomatik decoder testi ve ticari dosya paketi olusur.
5. **Blok 13:** `qr_fusion_delivery.zip` indirilir. Teslimden once final PNG'yi telefonda manuel okut.

## Ticari paketler

- **Basic:** temiz markali QR, final PNG/JPG, clean QR ve test raporu.
- **Standard:** ek varyasyonlar, print-ready PNG/PDF, sosyal onizleme ve clean SVG.
- **Premium:** logo-aware halftone final, layout pack, transparent PNG, musteri teslim brief'i ve gig readiness checklist.

## Portfoy ornekleri

Blok 5'teki `portfolio_example` alaniyla su 10 portfoy tipini tek tek uretebilirsin: restaurant, Instagram/Linktree, business card, product label, luxury brand, event ticket, colorful logo, black-white clean, halftone logo, social media preview.

Not: En iyi sonuc icin temiz, yuksek cozunurluklu, kontrastli logo kullan. Cok uzun URL yerine kisa URL kullanmak QR karmasikligini azaltir ve logoyu daha net gosterir. Artistic/logo-fused final yuksek cozunurluklu raster ciktidir; clean base QR ayrica SVG olarak verilebilir.



<!-- artqr-doc -->
## 1. Ortam Kurulumu

Colab ortaminda gerekli sistem paketleri ve Python kutuphaneleri kurulur. QR okuma icin `libzbar0`, gorsel isleme icin Pillow/OpenCV, QR uretimi icin `segno`, dogrulama icin `pyzbar` hazirlanir.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libzbar0
!pip -q install --upgrade "Pillow<12" opencv-python pyzbar segno numpy zxing-cpp


<!-- artqr-doc -->
## 2. Kutuphaneleri Yukleme

Notebook boyunca kullanilacak tum kutuphaneler tek yerde ice aktarilir. Bu hucre dosya yukleme, gorsel isleme, QR uretme, QR okuma ve ciktilari gostermek icin temel araclari hazirlar.


In [ ]:
import os
import cv2
import math
import json
import zipfile
import numpy as np
import segno
from PIL import Image, ImageEnhance, ImageFilter, ImageDraw, ImageOps, ImageFont
try:
    from pyzbar.pyzbar import decode as zbar_decode
except Exception as exc:
    zbar_decode = None
    print("pyzbar/zbar yuklenemedi; OpenCV + ZXing ile devam edilecek:", exc)

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    from google.colab import files
except Exception:
    files = None
    print("Google Colab files araci bulunamadi; yerel calismada upload/download hucrelerini manuel dosya yolu ile kullanin.")

try:
    import zxingcpp
except Exception as exc:
    zxingcpp = None
    print("ZXing yuklenemedi; pyzbar + OpenCV ile devam edilecek:", exc)


<!-- artqr-doc -->
## 3. Gorsel Yukleme

QR kodun icine islenecek ana gorsel bu hucrede yuklenir. Yuklenen dosya onizleme olarak gosterilir; boylece devam etmeden once dogru gorselin secildigini kontrol edebilirsiniz.


In [ ]:
#@title 3. Gorsel ve QR Giris Formu
# Gorsel ve QR giris modu
# upload_mode="generate_qr_from_data": QR, qr_data alanindan Segno ile uretilir.
# upload_mode="use_uploaded_qr_image": Elinizdeki hazir qr_code.png yuklenir ve fusion maskesi olarak kullanilir.
upload_mode = "generate_qr_from_data" #@param ["generate_qr_from_data", "use_uploaded_qr_image"]
upload_artwork_image = True #@param {type:"boolean"}
upload_qr_image_when_needed = True #@param {type:"boolean"}

artwork_path = ""
uploaded_qr_path = ""

if upload_artwork_image:
    print("Logo / artwork gorselini yukleyin.")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Artwork gorseli yuklenmedi.")
    artwork_path = list(uploaded.keys())[0]
else:
    artwork_path = "artwork.png" #@param {type:"string"}

if upload_mode == "use_uploaded_qr_image" and upload_qr_image_when_needed:
    print("Hazir QR gorselini yukleyin.")
    uploaded_qr = files.upload()
    if not uploaded_qr:
        raise ValueError("QR gorseli yuklenmedi.")
    uploaded_qr_path = list(uploaded_qr.keys())[0]
elif upload_mode == "use_uploaded_qr_image":
    uploaded_qr_path = "qr_code.png" #@param {type:"string"}

print("Yuklenen artwork:", artwork_path)
if uploaded_qr_path:
    print("Yuklenen QR:", uploaded_qr_path)

artwork_preview = Image.open(artwork_path).convert("RGBA")
display(artwork_preview)


<!-- artqr-doc -->
## 4. Ana QR ve Gorsel Ayarlari

QR icine yazilacak veri, cikti boyutu, hata toleransi, kenar boslugu ve gorselin kare alana nasil oturtulacagi burada belirlenir.

**Kalite mantigi:** `en_yuksek_kalite` modu hizdan feragat eder; daha buyuk cikti, daha fazla aday, daha sik guc taramasi, daha guclu finder bolgesi ve daha net modul kontrasti kullanir. Amac Fiverr ornegindeki gibi logonun onde gorundugu ama QR okuyucularin hala okuyabildigi temiz bir sonuc almaktir.


In [ ]:
#@title 4. Ana QR, Kalite Profili ve Fusion Ayarlari
#@markdown Bu blok QR verisini, cikti cozunurlugunu ve sanat/logonun QR ile ne kadar guclu karisacagini kontrol eder.
#@markdown `en_yuksek_kalite` en iyi gorsel/okunabilirlik dengesini arar; daha yavas calisir ama daha fazla aday test eder.

# Musteri paket seviyesi: Colab is akisini basit tutmak icin once bunu sec.
service_tier = "premium" #@param ["basic", "standard", "premium"]
apply_service_tier_to_quality = True #@param {type:"boolean"}

# QR verisi: Kod okutulunca acilacak metin/link. Hazir QR yuklersen expected_qr_data ile beklenen sonucu yazabilirsin.
# Logo uretimi yoktur; musteri hazir logo/artwork yukler, burada sadece QR icindeki bilgi uretilir.
qr_content_type = "url" #@param ["url", "plain_text", "email", "phone", "sms", "whatsapp", "wifi", "geo_location", "vcard", "calendar", "social_profile", "payment_link", "app_store", "online_menu", "google_review", "youtube", "raw"]
qr_data = "https://example.com" #@param {type:"string"}
expected_qr_data = "" #@param {type:"string"}

# Musteriden gelen bilgi alanlari. Sadece secilen qr_content_type icin gerekenleri doldurman yeterli.
contact_name = "" #@param {type:"string"}
contact_company = "" #@param {type:"string"}
contact_title = "" #@param {type:"string"}
contact_phone = "" #@param {type:"string"}
contact_email = "" #@param {type:"string"}
contact_website = "" #@param {type:"string"}
contact_address = "" #@param {type:"string"}
message_subject = "" #@param {type:"string"}
message_body = "" #@param {type:"string"}
wifi_ssid = "" #@param {type:"string"}
wifi_password = "" #@param {type:"string"}
wifi_security = "WPA" #@param ["WPA", "WEP", "nopass"]
wifi_hidden = False #@param {type:"boolean"}
geo_latitude = "" #@param {type:"string"}
geo_longitude = "" #@param {type:"string"}
event_title = "" #@param {type:"string"}
event_start = "" #@param {type:"string"}
event_end = "" #@param {type:"string"}
event_location = "" #@param {type:"string"}

def escape_qr_field(value):
    value = str(value or "")
    value = value.replace("\\", "\\\\")
    value = value.replace(";", "\\;").replace(",", "\\,")
    return value.replace("\n", "\\n")

def build_customer_qr_payload():
    base = qr_data.strip()
    if upload_mode == "use_uploaded_qr_image" or qr_content_type == "raw":
        return base
    if qr_content_type == "url":
        return base
    if qr_content_type == "plain_text":
        return base
    if qr_content_type in ("social_profile", "payment_link", "app_store", "online_menu", "google_review", "youtube"):
        # Bu tipler pratikte URL/link payload'udur; formda musteri dilinde ayrilir.
        return base
    if qr_content_type == "email":
        email = contact_email.strip() or base
        subject = message_subject.strip()
        body = message_body.strip()
        query = []
        if subject:
            query.append("subject=" + subject.replace(" ", "%20"))
        if body:
            query.append("body=" + body.replace(" ", "%20"))
        return "mailto:" + email + (("?" + "&".join(query)) if query else "")
    if qr_content_type == "phone":
        return "tel:" + (contact_phone.strip() or base)
    if qr_content_type == "sms":
        phone = contact_phone.strip() or base
        body = message_body.strip()
        return "SMSTO:" + phone + (":" + body if body else "")
    if qr_content_type == "whatsapp":
        phone = (contact_phone.strip() or base).replace("+", "").replace(" ", "")
        text = message_body.strip().replace(" ", "%20")
        return "https://wa.me/" + phone + (("?text=" + text) if text else "")
    if qr_content_type == "wifi":
        auth = "" if wifi_security == "nopass" else wifi_security
        return f"WIFI:T:{auth};S:{escape_qr_field(wifi_ssid)};P:{escape_qr_field(wifi_password)};H:{str(wifi_hidden).lower()};;"
    if qr_content_type == "geo_location":
        lat = geo_latitude.strip()
        lon = geo_longitude.strip()
        return f"geo:{lat},{lon}" if lat and lon else base
    if qr_content_type == "vcard":
        lines = [
            "BEGIN:VCARD",
            "VERSION:3.0",
            "FN:" + (contact_name.strip() or base),
        ]
        if contact_company.strip(): lines.append("ORG:" + contact_company.strip())
        if contact_title.strip(): lines.append("TITLE:" + contact_title.strip())
        if contact_phone.strip(): lines.append("TEL:" + contact_phone.strip())
        if contact_email.strip(): lines.append("EMAIL:" + contact_email.strip())
        if contact_website.strip(): lines.append("URL:" + contact_website.strip())
        if contact_address.strip(): lines.append("ADR:;;" + contact_address.strip())
        lines.append("END:VCARD")
        return "\n".join(lines)
    if qr_content_type == "calendar":
        lines = ["BEGIN:VEVENT", "SUMMARY:" + (event_title.strip() or base)]
        if event_start.strip(): lines.append("DTSTART:" + event_start.strip())
        if event_end.strip(): lines.append("DTEND:" + event_end.strip())
        if event_location.strip(): lines.append("LOCATION:" + event_location.strip())
        lines.append("END:VEVENT")
        return "\n".join(lines)
    return base

qr_payload = build_customer_qr_payload()
qr_data = qr_payload

# Kalite profili: manual secersen alttaki slider degerleri aynen kullanilir.
quality_mode = "en_yuksek_kalite" #@param ["en_yuksek_kalite", "dengeli", "hizli_test", "manual"]
apply_quality_mode = True #@param {type:"boolean"}

# Cikti boyutu: 4096 baski/satis icin daha temiz; 2048 hizli test icin yeterli.
output_size = 4096 #@param {type:"slider", min:1024, max:4096, step:512}
error_level = "h" #@param ["h", "q", "m", "l"]
quiet_zone = 5 #@param {type:"slider", min:4, max:10, step:1}

# Logo-aware QR secimi: Segno'nun maske adaylarini logo siluetindeki koyu/acik dengeye gore dener.
# Bu, halftone finalde logonun QR ile daha dogal oturmasini saglar.
logo_aware_qr_mask_optimize = True #@param {type:"boolean"}
logo_aware_mask_trials = 8 #@param {type:"slider", min:1, max:8, step:1}

# Logo/artwork kare alana nasil yerlestirilecek. Logo kirpilmesin istiyorsan contain_pad kullan.
fit_mode = "cover_crop" #@param ["cover_crop", "contain_pad"]
pad_red = 255 #@param {type:"slider", min:0, max:255, step:1}
pad_green = 255 #@param {type:"slider", min:0, max:255, step:1}
pad_blue = 255 #@param {type:"slider", min:0, max:255, step:1}

# Fusion taramasi: daha kucuk step daha cok aday ve daha kaliteli secim demektir.
dark_strength_start = 0.22 #@param {type:"slider", min:0.05, max:0.80, step:0.01}
dark_strength_end = 0.94 #@param {type:"slider", min:0.10, max:0.95, step:0.01}
dark_strength_step = 0.02 #@param {type:"slider", min:0.01, max:0.10, step:0.01}
light_strength_ratio = 0.9 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
module_contrast = 0.85 #@param {type:"slider", min:0.00, max:1.00, step:0.05}
center_bias = 0.9 #@param {type:"slider", min:0.00, max:1.00, step:0.05}

# Okunabilirlik destekleri: finder kose halkalari ve quiet zone temizligi QR okuyucular icin kritik.
finder_boost = 2.35 #@param {type:"slider", min:1.00, max:2.50, step:0.05}
quiet_zone_boost = 0.9 #@param {type:"slider", min:0.00, max:1.00, step:0.05}
contrast_after = 1.18 #@param {type:"slider", min:1.00, max:1.80, step:0.02}
sharpness_after = 1.55 #@param {type:"slider", min:1.00, max:2.50, step:0.05}

# Ticari kabul: pyzbar, OpenCV, ZXing icinden en az bu kadar okuyucu dogru sonucu okumali.
min_decoder_passes = 2 #@param {type:"slider", min:1, max:3, step:1}
prefer_best_score_not_first_pass = True #@param {type:"boolean"}

save_all_candidates = True #@param {type:"boolean"}
show_candidate_previews = False #@param {type:"boolean"}
download_final = True #@param {type:"boolean"}

QUALITY_MODES = {
    "en_yuksek_kalite": dict(output_size=4096, error_level="h", quiet_zone=5, logo_aware_qr_mask_optimize=True, logo_aware_mask_trials=8, dark_strength_start=0.22, dark_strength_end=0.94, dark_strength_step=0.02, light_strength_ratio=0.90, module_contrast=0.85, center_bias=0.90, finder_boost=2.35, quiet_zone_boost=0.90, contrast_after=1.18, sharpness_after=1.55, min_decoder_passes=2, save_all_candidates=True, show_candidate_previews=False),
    "dengeli": dict(output_size=3072, error_level="h", quiet_zone=4, logo_aware_qr_mask_optimize=True, logo_aware_mask_trials=8, dark_strength_start=0.25, dark_strength_end=0.88, dark_strength_step=0.03, light_strength_ratio=0.85, module_contrast=0.75, center_bias=0.82, finder_boost=2.10, quiet_zone_boost=0.75, contrast_after=1.14, sharpness_after=1.40, min_decoder_passes=2, save_all_candidates=True, show_candidate_previews=False),
    "hizli_test": dict(output_size=2048, error_level="h", quiet_zone=4, logo_aware_qr_mask_optimize=False, logo_aware_mask_trials=1, dark_strength_start=0.30, dark_strength_end=0.86, dark_strength_step=0.07, light_strength_ratio=0.80, module_contrast=0.70, center_bias=0.75, finder_boost=2.00, quiet_zone_boost=0.65, contrast_after=1.10, sharpness_after=1.25, min_decoder_passes=1, save_all_candidates=False, show_candidate_previews=True),
}

SERVICE_TIER_QUALITY = {"basic": "hizli_test", "standard": "dengeli", "premium": "en_yuksek_kalite"}
if apply_service_tier_to_quality:
    quality_mode = SERVICE_TIER_QUALITY.get(service_tier, quality_mode)
    print("Paket kalite eslesmesi uygulandi:", service_tier, "->", quality_mode)

if apply_quality_mode and quality_mode != "manual":
    for key, value in QUALITY_MODES[quality_mode].items():
        globals()[key] = value
    print("Kalite profili uygulandi:", quality_mode)

expected_data = expected_qr_data.strip() or qr_payload
print("QR icerik tipi:", qr_content_type)
print("QR payload hazir:", qr_payload[:220] + ("..." if len(qr_payload) > 220 else ""))




<!-- artqr-doc -->
## 5. Ticari Stil ve Logo Ayarlari

Daha profesyonel teslim dosyalari icin logo, rozet, modul sekli, renk paleti, arka plan ve dosya formatlari burada ayarlanir.

**Fiverr tarzi logo-onde cikti icin onemli bolum:** `make_halftone_logo_qr`, `use_halftone_as_final`, `halftone_logo_forward`, `halftone_light_logo_alpha` ve `halftone_logo_cell_threshold`. Bu ayarlar logoyu normal resim gibi basmak yerine QR modullerine bolerek arti/nokta/piksel gorunumune cevirir.


In [ ]:
#@title 5. Musteri Brief, Ticari Stil ve Teslim Paketi
#@markdown GUNLUK KULLANIM: Blok 4'te link/icerik, bu blokta marka rengi/sekil/logo ayarlarini degistir.
#@markdown `full_output_explorer` seciliyken notebook tum okunabilir teslimleri ve tum gorsel varyasyonlari uretir.

# Paket seviyesi Blok 4'ten gelir. Burada sadece gorunur olsun diye tekrar okunur.
service_tier = globals().get("service_tier", "premium")
print("Aktif servis paketi:", service_tier)

# Teslim profili: manual secersen bu bloktaki slider degerleri aynen kalir.
delivery_quality_mode = "en_yuksek_kalite" #@param ["en_yuksek_kalite", "dengeli", "hizli_test", "manual"]
apply_delivery_quality_mode = True #@param {type:"boolean"}
apply_service_tier_to_delivery = True #@param {type:"boolean"}

# Genel teslim paketi ac/kapat.
commercial_pack_enabled = True #@param {type:"boolean"}

# Tek secenekle profesyonel tum ciktilar: kaliteyi premium yapar ve shape/renk/logo/halftone varyasyonlarini ZIP'e ekler.
delivery_variant_mode = "all_premium_variants" #@param ["single_best", "all_premium_variants"]
include_unreadable_visual_variants = True #@param {type:"boolean"}

# Rakiplerin Illustrator/Fiverr akisini kodla taklit eden ana secim.
# custom: alttaki tum ayarlari elle kullan.
# pixel_dot_matrix: logo QR hucrelerine nokta/piksel/arti gibi yedirilir.
# basic_logosuz_duz: logosuz, sade ve maksimum okunabilir QR.
# design_from_logo: musterinin logosundan renk alan, rozetli ve sosyal/print teslimli tasarim.
# line_variations: dikey/yatay cizgi modullu gorsel varyasyonlar da uretilir.
# full_output_explorer: kodun uretebildigi tum teslimleri ve gorsel varyasyonlari ZIP'e ekler.
# tumunu_uret: okunabilir premium paket ve varyasyonlar ZIP'e eklenir.
master_style_choice = "full_output_explorer" #@param ["full_output_explorer", "custom", "pixel_dot_matrix", "basic_logosuz_duz", "design_from_logo", "line_variations", "qrbtf_style_pack", "tumunu_uret"]

# Musteri brief bilgileri: bunlar teslim raporuna ve ZIP icindeki is ozeti dosyasina yazilir.
client_name = "" #@param {type:"string"}
order_reference = "" #@param {type:"string"}
requested_style_note = "" #@param {type:"string"}
revision_note = "" #@param {type:"string"}
usage_place = "web_and_print" #@param ["web_only", "print", "web_and_print", "menu", "business_card", "sticker", "product_label", "event_ticket", "social_media"]
qr_static_or_dynamic = "static_lifetime" #@param ["static_lifetime", "dynamic_link_provided_by_client"]

# Pazar yeri teslim ekleri: hazir logodan sonra profesyonel sunum, frame ve baski olculeri.
make_scan_me_frame = True #@param {type:"boolean"}
scan_me_text = "SCAN ME" #@param {type:"string"}
scan_frame_style = "rounded_brand" #@param ["rounded_brand", "minimal", "bold_label"]
print_size_preset = "2x2_inch" #@param ["2x2_inch", "3x3_inch", "4x4_inch", "custom"]
custom_print_width_in = 2.0 #@param {type:"number"}
custom_print_height_in = 2.0 #@param {type:"number"}
print_dpi = 300 #@param [300, 600]

# Musteri gorsel ayarlari: otomatik presetlerden sonra tekrar uygulanir.
# Gunluk islerde genelde sadece bu alanlari ve Blok 4 link/icerik alanlarini degistir.
customer_visual_override_enabled = True #@param {type:"boolean"}
customer_color_mode = "brand_color" #@param ["black_white", "brand_color", "two_color", "gradient", "image_based"]
customer_dark_hex = "#111111" #@param {type:"string"}
customer_brand_hex = "#111111" #@param {type:"string"}
customer_gradient_start_hex = "#111111" #@param {type:"string"}
customer_gradient_end_hex = "#1F6FEB" #@param {type:"string"}
customer_light_hex = "#FFFFFF" #@param {type:"string"}
customer_module_shape = "rounded" #@param ["square", "rounded", "dot", "diamond", "hexagon", "vertical_line", "horizontal_line", "diagonal_tl_br", "diagonal_tr_bl", "cross_line", "interlock_line", "radial_line", "soft_blob"]
customer_finder_style = "circle" #@param ["classic", "rounded", "circle", "planet"]
customer_background_style = "white" #@param ["white", "brand_tint", "transparent", "artwork"]
customer_logo_mode = "center_badge" #@param ["none", "center_badge", "center_transparent", "background_logo", "corner_small", "logo_under_qr", "qr_over_logo_soft"]
customer_logo_badge_shape = "square" #@param ["rounded_square", "circle", "square"]
customer_logo_size_ratio = 0.16 #@param {type:"slider", min:0.08, max:0.26, step:0.01}
customer_halftone_mark_shape = "plus" #@param ["dot", "square", "plus", "diamond", "star"]

# Portfoy ve musteri isi icin tek tik stil secimi. custom secilirse style_preset manuel kullanilir.
portfolio_example = "halftone_logo_qr" #@param ["custom", "restaurant_qr", "instagram_linktree_qr", "business_card_qr", "product_label_qr", "luxury_brand_qr", "event_ticket_qr", "colorful_logo_qr", "black_white_clean_qr", "halftone_logo_qr", "social_media_preview_qr"]

# Preset renk/sekil seti. custom secilirse alttaki renk ve sekil ayarlari korunur.
style_preset = "custom" #@param ["custom", "business_clean", "luxury", "colorful", "restaurant", "event", "product_label", "social_media", "black_white_clean", "halftone_logo"]

# Logo yerlesimi: Fiverr tarzi sonuc icin asil finali halftone uretir; bu ayar klasik rozetli varyasyonlar icindir.
logo_source_mode = "uploaded_artwork" #@param ["uploaded_artwork", "none"]
logo_mode = "center_badge" #@param ["none", "center_badge", "center_transparent", "background_logo", "corner_small", "logo_under_qr", "qr_over_logo_soft"]
logo_size_ratio = 0.16 #@param {type:"slider", min:0.08, max:0.26, step:0.01}
logo_badge_shape = "rounded_square" #@param ["rounded_square", "circle", "square"]
logo_badge_opacity = 0.96 #@param {type:"slider", min:0.40, max:1.00, step:0.05}
logo_badge_padding = 0.28 #@param {type:"slider", min:0.05, max:0.45, step:0.01}
logo_visual_strength = 0.90 #@param {type:"slider", min:0.20, max:1.00, step:0.02}
logo_auto_readability_guard = True #@param {type:"boolean"}
brand_text = "" #@param {type:"string"}
logo_mode_from_preset = False #@param {type:"boolean"}

# Logo kalite hazirligi: dusuk cozunurluklu logoyu temizler, buyutur ve keskinlestirir.
logo_prep_enabled = True #@param {type:"boolean"}
logo_upscale_engine = "opencv_lanczos" #@param ["none", "opencv_lanczos"]
logo_upscale_factor = 4 #@param {type:"slider", min:1, max:4, step:1}
logo_denoise = "medium" #@param ["none", "light", "medium"]
logo_sharpen = "strong" #@param ["none", "light", "medium", "strong"]

# QR modullerinin sekli ve rengi. Halftone finalde mark_shape daha etkili; bu ayarlar diger varyasyonlari etkiler.
module_shape = "rounded" #@param ["square", "rounded", "dot", "diamond", "hexagon", "vertical_line", "horizontal_line", "diagonal_tl_br", "diagonal_tr_bl", "cross_line", "interlock_line", "radial_line", "soft_blob"]
finder_style = "rounded" #@param ["classic", "rounded", "circle", "planet"]
color_mode = "brand_color" #@param ["black_white", "brand_color", "two_color", "gradient", "image_based"]
dark_hex = "#111111" #@param {type:"string"}
brand_hex = "#111111" #@param {type:"string"}
gradient_start_hex = "#111111" #@param {type:"string"}
gradient_end_hex = "#1F6FEB" #@param {type:"string"}
light_hex = "#FFFFFF" #@param {type:"string"}

# Zemin beyaz kalirsa QR okuma daha guvenlidir. artwork zemini daha sanatsal ama daha risklidir.
background_style = "white" #@param ["white", "brand_tint", "transparent", "artwork"]
brand_tint_strength = 0.10 #@param {type:"slider", min:0.00, max:0.35, step:0.01}

# Uretilecek dosyalar. Paket presetleri bu degerleri otomatik guvenli hale getirir.
make_soft_balanced_strong = True #@param {type:"boolean"}
make_styled_clean_qr = True #@param {type:"boolean"}
make_halftone_logo_qr = True #@param {type:"boolean"}
use_halftone_as_final = True #@param {type:"boolean"}
halftone_apply_logo_badge_to_final = False #@param {type:"boolean"}
halftone_mark_shape = "plus" #@param ["dot", "square", "plus", "diamond", "star"]
halftone_density = 0.94 #@param {type:"slider", min:0.40, max:1.00, step:0.02}
halftone_logo_strength = 0.96 #@param {type:"slider", min:0.20, max:1.00, step:0.02}
halftone_logo_forward = True #@param {type:"boolean"}
halftone_light_logo_alpha = 0.94 #@param {type:"slider", min:0.20, max:1.00, step:0.02}
halftone_logo_cell_threshold = 0.045 #@param {type:"slider", min:0.02, max:0.30, step:0.005}
halftone_background_dots = True #@param {type:"boolean"}
halftone_designer_finders = True #@param {type:"boolean"}

# Logo-aware halftone motoru: logo konturunu, ic dolgusunu ve dis halo etkisini ayri guclendirir.
halftone_logo_aware_mode = True #@param {type:"boolean"}
halftone_outline_boost = 0.64 #@param {type:"slider", min:0.00, max:1.00, step:0.02}
halftone_edge_weight = 1.20 #@param {type:"slider", min:0.00, max:1.50, step:0.05}
halftone_negative_cutout = 0.34 #@param {type:"slider", min:0.00, max:1.00, step:0.02}
halftone_safe_background_density = 0.78 #@param {type:"slider", min:0.30, max:0.95, step:0.02}
make_social_preview = True #@param {type:"boolean"}
make_print_png = True #@param {type:"boolean"}
make_print_pdf = True #@param {type:"boolean"}
make_transparent_png = True #@param {type:"boolean"}
make_jpg = True #@param {type:"boolean"}
make_svg = True #@param {type:"boolean"}
make_eps = False #@param {type:"boolean"}
make_layout_pack = True #@param {type:"boolean"}
make_test_report = True #@param {type:"boolean"}

PRESETS = {
    "business_clean": dict(module_shape="rounded", finder_style="rounded", color_mode="brand_color", brand_hex="#1F6FEB", background_style="white", logo_mode="center_badge"),
    "luxury": dict(module_shape="rounded", finder_style="rounded", color_mode="gradient", gradient_start_hex="#151515", gradient_end_hex="#B08D57", background_style="brand_tint", brand_hex="#B08D57", logo_mode="center_badge"),
    "colorful": dict(module_shape="soft_blob", finder_style="circle", color_mode="gradient", gradient_start_hex="#FF4D6D", gradient_end_hex="#1FDDFF", background_style="white", logo_mode="center_transparent"),
    "restaurant": dict(module_shape="rounded", finder_style="classic", color_mode="two_color", dark_hex="#1F1A17", brand_hex="#D1495B", background_style="brand_tint", logo_mode="center_badge"),
    "event": dict(module_shape="dot", finder_style="circle", color_mode="gradient", gradient_start_hex="#111111", gradient_end_hex="#7C3AED", background_style="white", logo_mode="corner_small"),
    "product_label": dict(module_shape="square", finder_style="classic", color_mode="black_white", background_style="white", logo_mode="logo_under_qr"),
    "social_media": dict(module_shape="rounded", finder_style="rounded", color_mode="image_based", background_style="artwork", logo_mode="qr_over_logo_soft"),
    "black_white_clean": dict(module_shape="square", finder_style="classic", color_mode="black_white", background_style="white", logo_mode="none"),
    "halftone_logo": dict(module_shape="rounded", finder_style="circle", color_mode="brand_color", background_style="white", logo_mode="center_badge", halftone_mark_shape="plus", use_halftone_as_final=True),
}

PORTFOLIO_EXAMPLES = {
    "restaurant_qr": dict(style_preset="restaurant", usage_place="menu", service_tier="standard"),
    "instagram_linktree_qr": dict(style_preset="social_media", usage_place="social_media", service_tier="standard"),
    "business_card_qr": dict(style_preset="business_clean", usage_place="business_card", service_tier="standard"),
    "product_label_qr": dict(style_preset="product_label", usage_place="product_label", service_tier="standard"),
    "luxury_brand_qr": dict(style_preset="luxury", usage_place="web_and_print", service_tier="premium"),
    "event_ticket_qr": dict(style_preset="event", usage_place="event_ticket", service_tier="standard"),
    "colorful_logo_qr": dict(style_preset="colorful", usage_place="web_and_print", service_tier="premium"),
    "black_white_clean_qr": dict(style_preset="black_white_clean", usage_place="print", service_tier="basic", make_halftone_logo_qr=False, use_halftone_as_final=False),
    "halftone_logo_qr": dict(style_preset="halftone_logo", usage_place="web_and_print", service_tier="premium"),
    "social_media_preview_qr": dict(style_preset="social_media", usage_place="social_media", service_tier="premium", make_social_preview=True),
}

DELIVERY_QUALITY_MODES = {
    "en_yuksek_kalite": dict(logo_upscale_factor=4, logo_denoise="medium", logo_sharpen="strong", module_shape="rounded", finder_style="circle", make_soft_balanced_strong=True, make_styled_clean_qr=True, make_halftone_logo_qr=True, use_halftone_as_final=True, halftone_mark_shape="plus", halftone_density=0.94, halftone_logo_strength=0.96, halftone_logo_forward=True, halftone_light_logo_alpha=0.94, halftone_logo_cell_threshold=0.035, halftone_background_dots=True, halftone_designer_finders=True, halftone_logo_aware_mode=True, halftone_outline_boost=0.64, halftone_edge_weight=1.20, halftone_negative_cutout=0.34, halftone_safe_background_density=0.78, make_print_png=True, make_print_pdf=True, make_jpg=True, make_svg=True, make_layout_pack=True, make_test_report=True, make_social_preview=True, make_transparent_png=True),
    "dengeli": dict(logo_upscale_factor=3, logo_denoise="light", logo_sharpen="medium", module_shape="rounded", finder_style="rounded", make_soft_balanced_strong=True, make_styled_clean_qr=True, make_halftone_logo_qr=True, use_halftone_as_final=True, halftone_mark_shape="plus", halftone_density=0.92, halftone_logo_strength=0.92, halftone_logo_forward=True, halftone_light_logo_alpha=0.88, halftone_logo_cell_threshold=0.055, halftone_background_dots=True, halftone_designer_finders=True, halftone_logo_aware_mode=True, halftone_outline_boost=0.58, halftone_edge_weight=0.85, halftone_negative_cutout=0.48, halftone_safe_background_density=0.58, make_print_png=True, make_print_pdf=True, make_jpg=True, make_svg=True, make_layout_pack=False, make_test_report=True, make_social_preview=True, make_transparent_png=False),
    "hizli_test": dict(logo_upscale_factor=2, logo_denoise="light", logo_sharpen="medium", module_shape="rounded", finder_style="rounded", make_soft_balanced_strong=False, make_styled_clean_qr=True, make_halftone_logo_qr=False, use_halftone_as_final=False, halftone_mark_shape="plus", halftone_density=0.88, halftone_logo_strength=0.86, halftone_logo_forward=True, halftone_light_logo_alpha=0.78, halftone_logo_cell_threshold=0.08, halftone_background_dots=True, halftone_designer_finders=True, halftone_logo_aware_mode=True, halftone_outline_boost=0.42, halftone_edge_weight=0.65, halftone_negative_cutout=0.34, halftone_safe_background_density=0.54, make_print_png=False, make_print_pdf=False, make_jpg=True, make_svg=False, make_layout_pack=False, make_test_report=True, make_social_preview=False, make_transparent_png=False),
}

SERVICE_TIER_DELIVERY = {
    "basic": dict(delivery_quality_mode="hizli_test", make_soft_balanced_strong=False, make_styled_clean_qr=True, make_halftone_logo_qr=False, use_halftone_as_final=False, make_print_png=False, make_print_pdf=False, make_transparent_png=False, make_jpg=True, make_svg=False, make_layout_pack=False, make_social_preview=False, make_test_report=True),
    "standard": dict(delivery_quality_mode="dengeli", make_soft_balanced_strong=True, make_styled_clean_qr=True, make_halftone_logo_qr=True, use_halftone_as_final=True, make_print_png=True, make_print_pdf=True, make_transparent_png=False, make_jpg=True, make_svg=True, make_layout_pack=False, make_social_preview=True, make_test_report=True),
    "premium": dict(delivery_quality_mode="en_yuksek_kalite", make_soft_balanced_strong=True, make_styled_clean_qr=True, make_halftone_logo_qr=True, use_halftone_as_final=True, halftone_apply_logo_badge_to_final=False, make_print_png=True, make_print_pdf=True, make_transparent_png=True, make_jpg=True, make_svg=True, make_layout_pack=True, make_social_preview=True, make_test_report=True),
}


MASTER_STYLE_PRESETS = {
    "pixel_dot_matrix": dict(
        portfolio_example="halftone_logo_qr", style_preset="halftone_logo", service_tier="premium",
        delivery_variant_mode="single_best", module_shape="dot", finder_style="circle", logo_mode="center_badge",
        make_halftone_logo_qr=True, use_halftone_as_final=True, halftone_mark_shape="plus",
        halftone_density=0.96, halftone_logo_strength=0.96, halftone_logo_forward=True, make_social_preview=True,
    ),
    "basic_logosuz_duz": dict(
        portfolio_example="black_white_clean_qr", style_preset="black_white_clean", service_tier="basic",
        delivery_variant_mode="single_best", logo_source_mode="none", logo_mode="none", module_shape="square",
        finder_style="classic", color_mode="black_white", background_style="white", make_halftone_logo_qr=False,
        use_halftone_as_final=False, make_soft_balanced_strong=False, make_styled_clean_qr=True,
    ),
    "design_from_logo": dict(
        portfolio_example="colorful_logo_qr", style_preset="colorful", service_tier="premium",
        delivery_variant_mode="all_premium_variants", logo_source_mode="uploaded_artwork", logo_mode="center_transparent",
        color_mode="image_based", background_style="white", make_halftone_logo_qr=True, use_halftone_as_final=True,
        make_social_preview=True, make_layout_pack=True,
    ),
    "line_variations": dict(
        portfolio_example="custom", style_preset="business_clean", service_tier="premium",
        delivery_variant_mode="all_premium_variants", module_shape="vertical_line", finder_style="rounded",
        color_mode="brand_color", make_halftone_logo_qr=True, use_halftone_as_final=False,
        make_styled_clean_qr=True, make_social_preview=True,
    ),
    "qrbtf_style_pack": dict(
        portfolio_example="custom", style_preset="business_clean", service_tier="premium",
        delivery_variant_mode="all_premium_variants", module_shape="interlock_line", finder_style="planet",
        color_mode="brand_color", make_soft_balanced_strong=True, make_styled_clean_qr=True,
        make_halftone_logo_qr=True, use_halftone_as_final=False, make_svg=True, make_social_preview=True,
    ),
    "tumunu_uret": dict(
        portfolio_example="halftone_logo_qr", style_preset="halftone_logo", service_tier="premium",
        delivery_variant_mode="all_premium_variants", logo_source_mode="uploaded_artwork", logo_mode="center_badge",
        make_soft_balanced_strong=True, make_styled_clean_qr=True, make_halftone_logo_qr=True,
        use_halftone_as_final=True, make_print_png=True, make_print_pdf=True, make_transparent_png=True,
        make_jpg=True, make_svg=True, make_eps=True, make_layout_pack=True, make_social_preview=True, make_test_report=True,
    ),
    "full_output_explorer": dict(
        portfolio_example="halftone_logo_qr", style_preset="halftone_logo", service_tier="premium",
        delivery_variant_mode="all_premium_variants", include_unreadable_visual_variants=True,
        logo_source_mode="uploaded_artwork", logo_mode="center_badge", logo_badge_shape="square",
        make_soft_balanced_strong=True, make_styled_clean_qr=True, make_halftone_logo_qr=True,
        use_halftone_as_final=True, halftone_apply_logo_badge_to_final=False,
        make_print_png=True, make_print_pdf=True, make_transparent_png=True, make_jpg=True,
        make_svg=True, make_eps=True, make_layout_pack=True, make_social_preview=True, make_test_report=True,
    ),
}

if master_style_choice != "custom":
    for key, value in MASTER_STYLE_PRESETS[master_style_choice].items():
        globals()[key] = value
    print("Ana stil secimi uygulandi:", master_style_choice)

enable_all_delivery_variants = delivery_variant_mode == "all_premium_variants"
if enable_all_delivery_variants:
    service_tier = "premium"
    delivery_quality_mode = "en_yuksek_kalite"
    commercial_pack_enabled = True
    apply_delivery_quality_mode = True
    apply_service_tier_to_delivery = True
    make_soft_balanced_strong = True
    make_styled_clean_qr = True
    make_halftone_logo_qr = True
    use_halftone_as_final = True
    halftone_apply_logo_badge_to_final = False
    make_print_png = True
    make_print_pdf = True
    make_transparent_png = True
    make_jpg = True
    make_svg = True
    make_eps = True
    make_layout_pack = True
    make_social_preview = True
    make_test_report = True
    if master_style_choice == "full_output_explorer":
        include_unreadable_visual_variants = True
    print("Tum premium varyasyon modu aktif: kalite ve teslim formati otomatik genisletildi.")

if portfolio_example != "custom":
    for key, value in PORTFOLIO_EXAMPLES[portfolio_example].items():
        globals()[key] = value
    print("Portfoy ornegi uygulandi:", portfolio_example, "->", style_preset, "| paket:", service_tier)

if apply_service_tier_to_delivery:
    for key, value in SERVICE_TIER_DELIVERY.get(service_tier, {}).items():
        globals()[key] = value
    print("Paket teslim ayarlari uygulandi:", service_tier, "->", delivery_quality_mode)

if apply_delivery_quality_mode and delivery_quality_mode != "manual":
    for key, value in DELIVERY_QUALITY_MODES[delivery_quality_mode].items():
        globals()[key] = value
    print("Teslim kalite profili uygulandi:", delivery_quality_mode)

# Paket ayarlari kalite profilinden sonra tekrar uygulanir; boylece Basic/Standard/Premium teslim kapsami korunur.
if apply_service_tier_to_delivery:
    for key, value in SERVICE_TIER_DELIVERY.get(service_tier, {}).items():
        if key != "delivery_quality_mode":
            globals()[key] = value

if style_preset != "custom":
    selected_logo_mode = logo_mode
    for key, value in PRESETS[style_preset].items():
        if key == "logo_mode" and not logo_mode_from_preset:
            continue
        globals()[key] = value
    if not logo_mode_from_preset:
        logo_mode = selected_logo_mode
    print("Preset uygulandi:", style_preset, "| aktif logo_mode:", logo_mode)

if customer_visual_override_enabled:
    color_mode = customer_color_mode
    dark_hex = customer_dark_hex
    brand_hex = customer_brand_hex
    gradient_start_hex = customer_gradient_start_hex
    gradient_end_hex = customer_gradient_end_hex
    light_hex = customer_light_hex
    module_shape = customer_module_shape
    finder_style = customer_finder_style
    background_style = customer_background_style
    logo_mode = customer_logo_mode
    logo_badge_shape = customer_logo_badge_shape
    logo_size_ratio = customer_logo_size_ratio
    halftone_mark_shape = customer_halftone_mark_shape
    print("Musteri gorsel ayarlari uygulandi:", {
        "color_mode": color_mode,
        "brand_hex": brand_hex,
        "module_shape": module_shape,
        "finder_style": finder_style,
        "background_style": background_style,
        "logo_mode": logo_mode,
        "logo_badge_shape": logo_badge_shape,
        "halftone_mark_shape": halftone_mark_shape,
    })

OUTPUT_CATALOG = [
    "final_readable_art_qr.png",
    "clean_qr.png",
    "commercial/art_qr_soft.png",
    "commercial/art_qr_balanced.png",
    "commercial/art_qr_strong.png",
    "commercial/styled_clean_qr.png",
    "commercial/variant_clean_*.png",
    "commercial/variant_qrbtf_*.png",
    "commercial/logo_placement_*.png",
    "commercial/halftone_logo_qr.png",
    "commercial/halftone_*_*.png",
    "commercial/transparent_logo_qr.png",
    "commercial/print_ready_*.png",
    "commercial/print_ready_*.pdf",
    "commercial/final_scan_me_frame.png",
    "commercial/social_preview_1080.png",
    "commercial/*_layout.png",
    "commercial/clean_qr_vector.svg",
    "commercial/clean_qr_vector.eps",
    "commercial/qr_test_report.txt",
    "commercial/client_quality_summary.txt",
    "commercial/shape_quality_diagnosis.txt",
    "qr_fusion_delivery.zip",
]
print("Cikti katalogu aktif. Blok 12 ve 13 sonunda su tipler uretilir:")
for item in OUTPUT_CATALOG:
    print("-", item)

print("Musteri/brief ozeti:", {
    "client_name": client_name,
    "order_reference": order_reference,
    "usage_place": usage_place,
    "qr_content_type": globals().get("qr_content_type", ""),
    "qr_payload_preview": globals().get("qr_payload", "")[:160],
    "requested_style_note": requested_style_note,
})

print("Teslim kapsami:", {
    "master_style_choice": master_style_choice,
    "tier": service_tier,
    "portfolio_example": portfolio_example,
    "style_preset": style_preset,
    "print_pdf": make_print_pdf,
    "clean_svg": make_svg,
    "halftone_final": make_halftone_logo_qr and use_halftone_as_final,
    "social_preview": make_social_preview,
    "layout_pack": make_layout_pack,
    "scan_me_frame": make_scan_me_frame,
    "variant_mode": delivery_variant_mode,
    "all_variants": enable_all_delivery_variants,
})



<!-- artqr-doc -->
## 6. Gorsel Hazirlama ve QR Test Fonksiyonlari

Bu yardimci fonksiyonlar yuklenen gorseli kare formata getirir, temiz QR kodu olusturur ve uretilen dosyalarin gercekten okunup okunmadigini test eder. Notebook'un guvenli calismasi icin temel altyapi bu hucrededir.


In [ ]:
# Bu blok temel girdi hazirligini yapar: artwork kareye yerlestirilir, temiz QR uretilir ve QR/finder/quiet-zone maskeleri hesaplanir.
def prepare_artwork(path, size, fit_mode="cover_crop", pad_color=(255, 255, 255)):
    img = Image.open(path).convert("RGB")
    w, h = img.size

    if fit_mode == "cover_crop":
        scale = size / min(w, h)
        nw, nh = int(round(w * scale)), int(round(h * scale))
        img = img.resize((nw, nh), Image.Resampling.LANCZOS)
        left = (nw - size) // 2
        top = (nh - size) // 2
        img = img.crop((left, top, left + size, top + size))
        return img

    scale = size / max(w, h)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    img = img.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), pad_color)
    canvas.paste(img, ((size - nw) // 2, (size - nh) // 2))
    return canvas

def make_qr_png(data, path, size, error="h", border=4):
    qr = segno.make(data, error=error, boost_error=True)
    modules_with_border = qr.symbol_size(scale=1, border=border)[0]
    scale = max(1, size // modules_with_border)
    render_size = modules_with_border * scale
    qr.save(path, kind="png", scale=scale, border=border, dark="black", light="white")
    img = Image.open(path).convert("L")
    if img.size != (size, size):
        img = img.resize((size, size), Image.Resampling.NEAREST)
        img.save(path)
    return qr, img

def get_qr_masks(qr_img):
    arr = np.array(qr_img.convert("L"))
    dark_mask = arr < 128
    light_mask = ~dark_mask
    return dark_mask, light_mask

def build_finder_mask(size, qr, border):
    if qr is None:
        return np.zeros((size, size), dtype=bool)
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    qr_size_no_border = modules_total - 2 * border

    finder_mask = np.zeros((size, size), dtype=bool)

    # Finder pattern: QR okuyucunun once buldugu 3 buyuk kose karesi.
    # 7 modul ana kare + 1 modul separator bolgesi dahil edilerek guclendirilir.
    zones = [
        (border - 1, border - 1),
        (border + qr_size_no_border - 8, border - 1),
        (border - 1, border + qr_size_no_border - 8),
    ]

    for mx, my in zones:
        x1 = max(0, int(math.floor(mx * module_px)))
        y1 = max(0, int(math.floor(my * module_px)))
        x2 = min(size, int(math.ceil((mx + 9) * module_px)))
        y2 = min(size, int(math.ceil((my + 9) * module_px)))
        finder_mask[y1:y2, x1:x2] = True

    return finder_mask

def build_quiet_zone_mask(size, qr, border):
    if qr is None:
        b = max(1, int(round(size * 0.06)))
        mask = np.zeros((size, size), dtype=bool)
        mask[:b, :] = True
        mask[-b:, :] = True
        mask[:, :b] = True
        mask[:, -b:] = True
        return mask
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    b = int(round(border * module_px))
    mask = np.zeros((size, size), dtype=bool)
    mask[:b, :] = True
    mask[-b:, :] = True
    mask[:, :b] = True
    mask[:, -b:] = True
    return mask

def build_module_center_weight(size, qr, border, center_bias):
    if qr is None or center_bias <= 0:
        return np.ones((size, size), dtype=np.float32)

    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total

    yy, xx = np.indices((size, size))
    mx = (xx / module_px) % 1.0
    my = (yy / module_px) % 1.0

    dx = np.abs(mx - 0.5) * 2.0
    dy = np.abs(my - 0.5) * 2.0
    dist = np.maximum(dx, dy)
    center = np.clip(1.0 - dist, 0.0, 1.0)
    return (1.0 - center_bias) + center_bias * center

def luminance_fusion(
    art_img,
    qr_img,
    qr,
    border,
    dark_strength,
    light_ratio,
    finder_boost,
    quiet_boost,
    module_contrast=0.35,
    center_bias=0.65,
):
    art = np.array(art_img.convert("RGB"))
    dark_mask, light_mask = get_qr_masks(qr_img)
    finder_mask = build_finder_mask(art.shape[0], qr, border)
    quiet_mask = build_quiet_zone_mask(art.shape[0], qr, border)
    center_weight = build_module_center_weight(art.shape[0], qr, border, center_bias)

    lab = cv2.cvtColor(art, cv2.COLOR_RGB2LAB).astype(np.float32)
    l = lab[:, :, 0]

    strength_map = np.full(l.shape, dark_strength, dtype=np.float32)
    strength_map *= center_weight
    strength_map[finder_mask] *= finder_boost
    strength_map = np.clip(strength_map, 0.0, 0.95)

    dark_floor = 255.0 * (1.0 - module_contrast)
    light_floor = 255.0 * module_contrast

    dark_target = np.minimum(l * (1.0 - strength_map), dark_floor)
    light_strength = np.clip(strength_map * light_ratio, 0.0, 0.90)
    light_target = np.maximum(l + (255.0 - l) * light_strength, light_floor)

    l2 = l.copy()
    l2[dark_mask] = dark_target[dark_mask]
    l2[light_mask] = light_target[light_mask]

    if quiet_boost > 0:
        l2[quiet_mask] = l2[quiet_mask] + (255.0 - l2[quiet_mask]) * quiet_boost

    lab[:, :, 0] = np.clip(l2, 0, 255)
    fused = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
    return Image.fromarray(fused)

def polish_image(img, contrast=1.08, sharpness=1.25):
    img = ImageEnhance.Contrast(img).enhance(contrast)
    img = ImageEnhance.Sharpness(img).enhance(sharpness)
    img = img.filter(ImageFilter.UnsharpMask(radius=1.0, percent=90, threshold=3))
    return img


def analyze_logo_quality(path):
    img = Image.open(path).convert("RGBA")
    rgb = img.convert("RGB")
    arr = np.array(rgb)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    alpha = np.array(img.getchannel("A"))
    warnings = []
    w, h = img.size
    if min(w, h) < 600:
        warnings.append("Cozunurluk dusuk; logo_upscale_factor 2-4 onerilir.")
    if alpha.min() < 255:
        transparent_ratio = float((alpha < 250).mean())
        if transparent_ratio > 0.10:
            warnings.append("Transparan alan yuksek; badge veya beyaz zemin daha guvenli olabilir.")
    contrast = float(gray.std())
    if contrast < 32:
        warnings.append("Kontrast zayif; module_contrast/finder_boost veya sharpen artirilabilir.")
    edges = cv2.Laplacian(gray, cv2.CV_64F).var()
    if edges < 80:
        warnings.append("Gorsel yumusak/bulanik; sharpen medium/strong onerilir.")
    small_detail_ratio = float((cv2.Canny(gray, 80, 160) > 0).mean())
    if small_detail_ratio > 0.18:
        warnings.append("Cok ince detay/kucuk yazi var; QR icinde kaybolabilir.")
    return {
        "size": img.size,
        "contrast": round(contrast, 2),
        "sharpness": round(float(edges), 2),
        "transparent_ratio": round(float((alpha < 250).mean()), 3),
        "small_detail_ratio": round(small_detail_ratio, 3),
        "warnings": warnings,
    }

def prepare_logo_source(path, out_path="qr_fusion_outputs/prepared_logo_source.png"):
    img = Image.open(path).convert("RGBA")
    if not logo_prep_enabled:
        img.save(out_path)
        return out_path
    if logo_upscale_engine in ("opencv_lanczos", "real_esrgan") and logo_upscale_factor > 1:
        if logo_upscale_engine == "real_esrgan":
            print("Real-ESRGAN model entegrasyonu bu notebookta opsiyonel hazirlik anahtari olarak duruyor; OpenCV Lanczos fallback kullaniliyor.")
        requested_factor = int(logo_upscale_factor)
        # 4096px ve ustu kaynaklarda 4x upscale 16K gorsel uretip Pillow guvenlik limitine takilir.
        max_side = max(int(output_size), 4096)
        scale = min(requested_factor, max_side / max(img.size))
        if scale > 1.01:
            new_size = (int(round(img.size[0] * scale)), int(round(img.size[1] * scale)))
            img = img.resize(new_size, Image.Resampling.LANCZOS)
        else:
            print(f"Logo zaten yeterli cozunurlukte ({img.size[0]}x{img.size[1]}); {requested_factor}x upscale atlandi.")
    if logo_denoise != "none":
        strength = 3 if logo_denoise == "light" else 7
        arr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
        arr = cv2.fastNlMeansDenoisingColored(arr, None, strength, strength, 7, 21)
        rgb = Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)).convert("RGBA")
        rgb.putalpha(img.getchannel("A"))
        img = rgb
    sharpen_map = {"none": 1.0, "light": 1.15, "medium": 1.35, "strong": 1.65}
    if sharpen_map.get(logo_sharpen, 1.0) > 1.0:
        rgb = ImageEnhance.Sharpness(img.convert("RGB")).enhance(sharpen_map[logo_sharpen])
        rgb = rgb.filter(ImageFilter.UnsharpMask(radius=1.0, percent=110, threshold=3))
        rgb = rgb.convert("RGBA")
        rgb.putalpha(img.getchannel("A"))
        img = rgb
    img.save(out_path)
    return out_path

def prepare_fullscreen_logo_artwork(
    path,
    out_path="qr_fusion_outputs/prepared_logo_fullscreen.png",
    size=None,
    fill_ratio=0.88,
    posterize_bits=4,
    edge_boost=1.0,
):
    """Logoyu QR halftone icin tam ekran, kontrastli ve posterize ana gorsele cevirir."""
    size = int(size or output_size)
    img = Image.open(path).convert("RGBA")
    alpha = np.array(img.getchannel("A"))
    rgb = np.array(img.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    color_delta = np.max(np.abs(rgb.astype(np.int16) - 255), axis=2)

    if float((alpha > 24).mean()) < 0.96:
        logo_mask = alpha > 24
    else:
        logo_mask = (alpha > 24) & ((gray < 246) | (color_delta > 14))
    logo_mask = cv2.morphologyEx(logo_mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)) > 0

    ys, xs = np.where(logo_mask)
    if xs.size and ys.size:
        pad = max(2, int(round(max(img.size) * 0.015)))
        left = max(0, int(xs.min()) - pad)
        top = max(0, int(ys.min()) - pad)
        right = min(img.size[0], int(xs.max()) + pad + 1)
        bottom = min(img.size[1], int(ys.max()) + pad + 1)
        img = img.crop((left, top, right, bottom))

    target = max(1, int(round(size * float(np.clip(fill_ratio, 0.62, 0.96)))))
    fitted = ImageOps.contain(img, (target, target), Image.Resampling.LANCZOS)
    layer = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    layer.paste(fitted, ((size - fitted.size[0]) // 2, (size - fitted.size[1]) // 2), fitted)

    alpha = np.array(layer.getchannel("A"))
    rgb_img = layer.convert("RGB")
    rgb_img = ImageEnhance.Contrast(rgb_img).enhance(1.45)
    rgb_img = ImageEnhance.Color(rgb_img).enhance(1.18)
    rgb_img = ImageEnhance.Sharpness(rgb_img).enhance(1.35)
    rgb_img = ImageOps.posterize(rgb_img, int(np.clip(posterize_bits, 3, 6)))

    if edge_boost > 0:
        kernel_size = max(3, int(round(size / 520)))
        if kernel_size % 2 == 0:
            kernel_size += 1
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        strong_alpha = alpha > 24
        dilated = cv2.dilate(strong_alpha.astype(np.uint8) * 255, kernel, iterations=max(1, int(round(edge_boost))))
        edge = (dilated > 0) & ~strong_alpha
        edge_layer = Image.new("RGBA", (size, size), hex_to_rgb(globals().get("dark_hex", "#111111")) + (0,))
        edge_alpha = np.zeros((size, size), dtype=np.uint8)
        edge_alpha[edge] = 185
        edge_layer.putalpha(Image.fromarray(edge_alpha, mode="L"))
        poster_layer = rgb_img.convert("RGBA")
        poster_layer.putalpha(Image.fromarray(np.maximum(alpha, dilated).astype(np.uint8), mode="L"))
        canvas = Image.new("RGBA", (size, size), (255, 255, 255, 0))
        canvas.alpha_composite(edge_layer)
        canvas.alpha_composite(poster_layer)
    else:
        canvas = rgb_img.convert("RGBA")
        canvas.putalpha(Image.fromarray(alpha.astype(np.uint8), mode="L"))

    canvas.save(out_path)
    return out_path


def load_uploaded_qr_image(path, size):
    img = Image.open(path).convert("L")
    img = ImageOps.contain(img, (size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("L", (size, size), 255)
    canvas.paste(img, ((size - img.size[0]) // 2, (size - img.size[1]) // 2))
    canvas = canvas.point(lambda p: 0 if p < 160 else 255)
    return canvas

def test_qr_all(path, expected_data=None):
    pil_img = Image.open(path).convert("RGB")

    zbar_texts = []
    if zbar_decode is not None:
        try:
            zbar_results = zbar_decode(pil_img)
            zbar_texts = [r.data.decode("utf-8") for r in zbar_results]
        except Exception:
            zbar_texts = []

    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    detector = cv2.QRCodeDetector()
    cv_text, points, _ = detector.detectAndDecode(cv_img)
    cv_texts = [cv_text] if cv_text else []

    zxing_texts = []
    if zxingcpp is not None:
        try:
            zxing_results = zxingcpp.read_barcodes(pil_img)
            zxing_texts = [r.text for r in zxing_results if getattr(r, "text", None)]
        except Exception:
            zxing_texts = []

    engines = {"zbar": zbar_texts, "opencv": cv_texts, "zxing": zxing_texts}
    all_texts = list(dict.fromkeys(zbar_texts + cv_texts + zxing_texts))
    if expected_data:
        passed_engines = [name for name, texts in engines.items() if expected_data in texts]
    else:
        passed_engines = [name for name, texts in engines.items() if texts]
    ok = len(passed_engines) >= min_decoder_passes

    return {
        "ok": ok,
        "texts": all_texts,
        "passed_engines": passed_engines,
        "decoder_pass_count": len(passed_engines),
        "zbar": zbar_texts,
        "opencv": cv_texts,
        "zxing": zxing_texts,
    }

def qr_quality_score(path, result, strength=0.0):
    img = Image.open(path).convert("L")
    arr = np.array(img, dtype=np.float32)
    contrast = float(arr.std() / 64.0)
    readability = float(result.get("decoder_pass_count", 0)) * 10.0
    aesthetic_penalty = abs(float(strength) - 0.55) * 2.0
    return readability + min(contrast, 4.0) - aesthetic_penalty


<!-- artqr-doc -->
## 7. Stil Cizim Fonksiyonlari

Bu bolum renkleri donusturur, modul sekillerini cizer, finder alanlarini duzenler, logo/rozet ekler ve ticari gorunumleri olusturur. Yani QR kodun sadece okunur degil, ayni zamanda guzel gorunmesini saglayan gorsel motor burada calisir.


In [ ]:
# Bu blok ticari gorsel motorudur: renkler, modul sekilleri, logo-aware halftone, finder tasarimi, rapor ve teslim yardimcilari burada tanimlanir.
def hex_to_rgb(value):
    value = value.strip().lstrip("#")
    if len(value) == 3:
        value = "".join(ch * 2 for ch in value)
    if len(value) != 6:
        raise ValueError(f"Gecersiz hex renk: {value}")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

def blend_rgb(a, b, t):
    return tuple(int(round(a[i] * (1.0 - t) + b[i] * t)) for i in range(3))

def make_background(size, style, light_rgb, brand_rgb, artwork=None, transparent=False):
    if transparent:
        return Image.new("RGBA", (size, size), (255, 255, 255, 0))
    if style == "artwork" and artwork is not None:
        return artwork.convert("RGBA")
    if style == "brand_tint":
        rgb = blend_rgb(light_rgb, brand_rgb, brand_tint_strength)
        return Image.new("RGBA", (size, size), rgb + (255,))
    return Image.new("RGBA", (size, size), light_rgb + (255,))

def module_is_dark(qr_img, x1, y1, x2, y2):
    cx = min(qr_img.size[0] - 1, max(0, int(round((x1 + x2) / 2))))
    cy = min(qr_img.size[1] - 1, max(0, int(round((y1 + y2) / 2))))
    return qr_img.getpixel((cx, cy)) < 128

def draw_module(draw, box, shape, fill, radius_ratio=0.35):
    x1, y1, x2, y2 = box
    inset = max(1, int(round((x2 - x1) * 0.10)))
    box = (x1 + inset, y1 + inset, x2 - inset, y2 - inset)
    if shape == "dot":
        draw.ellipse(box, fill=fill)
    elif shape == "diamond":
        cx = (box[0] + box[2]) / 2
        cy = (box[1] + box[3]) / 2
        draw.polygon([(cx, box[1]), (box[2], cy), (cx, box[3]), (box[0], cy)], fill=fill)
    elif shape == "hexagon":
        w = box[2] - box[0]
        h = box[3] - box[1]
        draw.polygon([(box[0] + w*0.25, box[1]), (box[0] + w*0.75, box[1]), (box[2], box[1] + h*0.5), (box[0] + w*0.75, box[3]), (box[0] + w*0.25, box[3]), (box[0], box[1] + h*0.5)], fill=fill)
    elif shape == "vertical_line":
        cx = (box[0] + box[2]) / 2
        line_w = max(1, int((box[2] - box[0]) * 0.42))
        draw.rounded_rectangle((cx - line_w/2, box[1], cx + line_w/2, box[3]), radius=max(1, line_w//2), fill=fill)
    elif shape == "horizontal_line":
        cy = (box[1] + box[3]) / 2
        line_h = max(1, int((box[3] - box[1]) * 0.42))
        draw.rounded_rectangle((box[0], cy - line_h/2, box[2], cy + line_h/2), radius=max(1, line_h//2), fill=fill)
    elif shape in ("diagonal_tl_br", "diagonal_tr_bl", "cross_line", "interlock_line", "radial_line"):
        w = max(1, box[2] - box[0])
        line_w = max(1, int(w * 0.34))
        if shape == "diagonal_tl_br":
            draw.line((box[0], box[1], box[2], box[3]), fill=fill, width=line_w)
        elif shape == "diagonal_tr_bl":
            draw.line((box[2], box[1], box[0], box[3]), fill=fill, width=line_w)
        elif shape == "cross_line":
            draw.line((box[0], box[1], box[2], box[3]), fill=fill, width=max(1, int(line_w * 0.75)))
            draw.line((box[2], box[1], box[0], box[3]), fill=fill, width=max(1, int(line_w * 0.75)))
        elif shape == "radial_line":
            cx = (box[0] + box[2]) / 2
            cy = (box[1] + box[3]) / 2
            draw.line((cx, box[1], cx, box[3]), fill=fill, width=max(1, int(line_w * 0.72)))
            draw.line((box[0], cy, box[2], cy), fill=fill, width=max(1, int(line_w * 0.72)))
        else:
            # QRBTF interlock hissi: hucre merkezini hem yatay hem dikey baglayarak orgu efekti verir.
            cx = (box[0] + box[2]) / 2
            cy = (box[1] + box[3]) / 2
            arm = max(1, int(w * 0.30))
            draw.rounded_rectangle((box[0], cy - arm / 2, box[2], cy + arm / 2), radius=max(1, arm // 2), fill=fill)
            draw.rounded_rectangle((cx - arm / 2, box[1], cx + arm / 2, box[3]), radius=max(1, arm // 2), fill=fill)
    elif shape == "soft_blob":
        radius = max(1, int(round((box[2] - box[0]) * 0.48)))
        draw.rounded_rectangle(box, radius=radius, fill=fill)
    elif shape == "rounded":
        radius = max(1, int(round((box[2] - box[0]) * radius_ratio)))
        draw.rounded_rectangle(box, radius=radius, fill=fill)
    else:
        draw.rectangle(box, fill=fill)


def make_logo_mask(logo_path, size, threshold=245):
    logo = Image.open(logo_path).convert("RGBA")
    fitted = ImageOps.contain(logo, (int(size * 0.90), int(size * 0.90)), Image.Resampling.LANCZOS)
    layer = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    layer.paste(fitted, ((size - fitted.size[0]) // 2, (size - fitted.size[1]) // 2), fitted)
    alpha = np.array(layer.getchannel("A"))
    rgb = np.array(layer.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    color_delta = np.max(np.abs(rgb.astype(np.int16) - 255), axis=2)
    transparent_logo = np.mean(alpha > 24) < 0.96
    if transparent_logo:
        # Seffaf PNG/SVG logolarda beyaz logo da logodur; sadece "beyaz olmayan"
        # piksel ararsak Instagram/WhatsApp gibi beyaz isaretler kaybolur.
        mask = alpha > 24
    else:
        mask = (alpha > 24) & ((gray < threshold) | (color_delta > 18))
    mask = cv2.morphologyEx(mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)) > 0
    return layer, mask

def logo_sample_color(logo_layer, x1, y1, x2, y2, fallback):
    crop = logo_layer.crop((x1, y1, max(x1 + 1, x2), max(y1 + 1, y2))).convert("RGBA")
    arr = np.array(crop)
    alpha = arr[:, :, 3] > 24
    if not alpha.any():
        return fallback
    rgb = arr[:, :, :3][alpha]
    color = tuple(int(v) for v in np.median(rgb, axis=0))
    return color + (255,)

def draw_halftone_mark(draw, box, shape, fill, density=0.86):
    x1, y1, x2, y2 = box
    w = max(1, x2 - x1)
    h = max(1, y2 - y1)
    inset = int(round(min(w, h) * (1.0 - density) * 0.45))
    x1 += inset; y1 += inset; x2 -= inset; y2 -= inset
    if x2 <= x1 or y2 <= y1:
        return
    if shape == "dot":
        draw.ellipse((x1, y1, x2, y2), fill=fill)
    elif shape == "square":
        draw.rectangle((x1, y1, x2, y2), fill=fill)
    elif shape == "diamond":
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        draw.polygon([(cx, y1), (x2, cy), (cx, y2), (x1, cy)], fill=fill)
    elif shape == "star":
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        rx = (x2 - x1) * 0.48
        ry = (y2 - y1) * 0.48
        pts = []
        for i in range(8):
            r = 1.0 if i % 2 == 0 else 0.42
            a = -math.pi / 2 + i * math.pi / 4
            pts.append((cx + math.cos(a) * rx * r, cy + math.sin(a) * ry * r))
        draw.polygon(pts, fill=fill)
    else:
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        arm = max(1, int(round(min(x2 - x1, y2 - y1) * 0.28)))
        draw.rounded_rectangle((cx - arm / 2, y1, cx + arm / 2, y2), radius=max(1, arm // 2), fill=fill)
        draw.rounded_rectangle((x1, cy - arm / 2, x2, cy + arm / 2), radius=max(1, arm // 2), fill=fill)

def draw_designer_finder(draw, box, accent, light=(255, 255, 255, 255), dark=(18, 18, 18, 255)):
    x1, y1, x2, y2 = box
    size = max(1, x2 - x1)
    pad = max(2, int(round(size * 0.04)))
    radius = max(4, int(round(size * 0.08)))
    outer = (x1 + pad, y1 + pad, x2 - pad, y2 - pad)

    # Rakip orneklerdeki gibi finder bolgesini sade ve yuksek kontrastli tut:
    # okuyucu once burayi yakalar, gorsel kalite de buradan guven verir.
    draw.rounded_rectangle(outer, radius=radius, fill=dark)
    ring_pad = max(2, int(round(size * 0.15)))
    ring = (outer[0] + ring_pad, outer[1] + ring_pad, outer[2] - ring_pad, outer[3] - ring_pad)
    draw.rounded_rectangle(ring, radius=max(3, int(radius * 0.55)), fill=light)
    inner_pad = max(2, int(round(size * 0.25)))
    inner = (outer[0] + inner_pad, outer[1] + inner_pad, outer[2] - inner_pad, outer[3] - inner_pad)
    draw.ellipse(inner, fill=accent)
    core_pad = max(2, int(round(size * 0.35)))
    core = (outer[0] + core_pad, outer[1] + core_pad, outer[2] - core_pad, outer[3] - core_pad)
    draw.ellipse(core, fill=dark)

def finder_module_origins(modules_total, border):
    qr_size_no_border = modules_total - 2 * border
    return [
        (border, border),
        (border + qr_size_no_border - 7, border),
        (border, border + qr_size_no_border - 7),
    ]


def qr_version_from_modules(modules_total, border):
    qr_size_no_border = modules_total - 2 * border
    return max(1, int(round((qr_size_no_border - 17) / 4)))


def alignment_pattern_centers(version):
    # ISO QR alignment merkezleri. Version 1'de alignment pattern yoktur.
    table = {
        1: [], 2: [6, 18], 3: [6, 22], 4: [6, 26], 5: [6, 30], 6: [6, 34],
        7: [6, 22, 38], 8: [6, 24, 42], 9: [6, 26, 46], 10: [6, 28, 50],
        11: [6, 30, 54], 12: [6, 32, 58], 13: [6, 34, 62], 14: [6, 26, 46, 66],
        15: [6, 26, 48, 70], 16: [6, 26, 50, 74], 17: [6, 30, 54, 78],
        18: [6, 30, 56, 82], 19: [6, 30, 58, 86], 20: [6, 34, 62, 90],
        21: [6, 28, 50, 72, 94], 22: [6, 26, 50, 74, 98], 23: [6, 30, 54, 78, 102],
        24: [6, 28, 54, 80, 106], 25: [6, 32, 58, 84, 110], 26: [6, 30, 58, 86, 114],
        27: [6, 34, 62, 90, 118], 28: [6, 26, 50, 74, 98, 122],
        29: [6, 30, 54, 78, 102, 126], 30: [6, 26, 52, 78, 104, 130],
        31: [6, 30, 56, 82, 108, 134], 32: [6, 34, 60, 86, 112, 138],
        33: [6, 30, 58, 86, 114, 142], 34: [6, 34, 62, 90, 118, 146],
        35: [6, 30, 54, 78, 102, 126, 150], 36: [6, 24, 50, 76, 102, 128, 154],
        37: [6, 28, 54, 80, 106, 132, 158], 38: [6, 32, 58, 84, 110, 136, 162],
        39: [6, 26, 54, 82, 110, 138, 166], 40: [6, 30, 58, 86, 114, 142, 170],
    }
    return table.get(int(version), [])


def qr_structural_module_sets(modules_total, border):
    """QR'nin okunurluk iskeletini dekorasyondan ayirir.

    Rakip logo-QR islerinde guzel gorunen kisim veri modulleridir; finder,
    separator, timing, alignment ve format/version alanlari net kalir.
    """
    protected = set()
    finder_core = set()
    finder_guard = set()
    qr_n = modules_total - 2 * border
    version = qr_version_from_modules(modules_total, border)

    for fx, fy in finder_module_origins(modules_total, border):
        for yy in range(fy, fy + 7):
            for xx in range(fx, fx + 7):
                finder_core.add((xx, yy))
        for yy in range(fy - 1, fy + 8):
            for xx in range(fx - 1, fx + 8):
                if 0 <= xx < modules_total and 0 <= yy < modules_total:
                    finder_guard.add((xx, yy))
                    protected.add((xx, yy))

    # Timing patterns: 6. satir/sutun, finder aralarinda grid koordinatini tasir.
    timing_y = border + 6
    timing_x = border + 6
    for i in range(border, border + qr_n):
        protected.add((i, timing_y))
        protected.add((timing_x, i))

    # Format bilgisi finderlarin yanindadir; versiyon 7+ icin version bilgisi de korunur.
    for i in range(0, 9):
        protected.add((border + i, border + 8))
        protected.add((border + 8, border + i))
        protected.add((border + qr_n - 1 - i, border + 8))
        protected.add((border + 8, border + qr_n - 1 - i))
    if version >= 7:
        for yy in range(border, border + 6):
            for xx in range(border + qr_n - 11, border + qr_n - 8):
                protected.add((xx, yy))
        for yy in range(border + qr_n - 11, border + qr_n - 8):
            for xx in range(border, border + 6):
                protected.add((xx, yy))

    centers = [border + c for c in alignment_pattern_centers(version)]
    for cx in centers:
        for cy in centers:
            if (cx, cy) in [(border + 6, border + 6), (border + qr_n - 7, border + 6), (border + 6, border + qr_n - 7)]:
                continue
            for yy in range(cy - 2, cy + 3):
                for xx in range(cx - 2, cx + 3):
                    if 0 <= xx < modules_total and 0 <= yy < modules_total:
                        protected.add((xx, yy))

    return {
        "protected": protected,
        "finder_core": finder_core,
        "finder_guard": finder_guard,
        "timing": {(i, timing_y) for i in range(border, border + qr_n)} | {(timing_x, i) for i in range(border, border + qr_n)},
        "version": version,
    }


def draw_raw_qr_modules(draw, qr_img, modules, module_px, shape, dark, light=None):
    for mx, my in modules:
        x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
        x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
        if light is not None and not module_is_dark(qr_img, x1, y1, x2, y2):
            draw.rectangle((x1, y1, x2, y2), fill=light)
            continue
        if module_is_dark(qr_img, x1, y1, x2, y2):
            draw_module(draw, (x1, y1, x2, y2), shape, dark)


def logo_detail_maps(logo_path, size, threshold=245):
    """Logo maskesini satis kalitesi icin ayri okunur katmanlara ayirir."""
    logo_layer, logo_mask = make_logo_mask(logo_path, size, threshold=threshold)
    base = logo_mask.astype(np.uint8) * 255
    module_hint = max(3, size // 180)
    kernel = np.ones((module_hint, module_hint), np.uint8)
    tight = cv2.morphologyEx(base, cv2.MORPH_CLOSE, kernel) > 0
    eroded = cv2.erode(tight.astype(np.uint8) * 255, kernel, iterations=1) > 0
    dilated = cv2.dilate(tight.astype(np.uint8) * 255, kernel, iterations=1) > 0
    edge = np.logical_xor(dilated, eroded)
    halo = np.logical_and(dilated, ~tight)

    gray = np.array(logo_layer.convert("L"))
    alpha = np.array(logo_layer.getchannel("A")) > 24
    dark_detail = alpha & (gray < 150)
    light_detail = alpha & (gray > 218) & tight
    return {
        "layer": logo_layer,
        "mask": tight,
        "core": eroded,
        "edge": edge,
        "halo": halo,
        "dark_detail": dark_detail,
        "light_detail": light_detail,
    }


def cell_mean(mask, x1, y1, x2, y2):
    if x2 <= x1 or y2 <= y1:
        return 0.0
    return float(mask[y1:y2, x1:x2].mean())


def qr_dark_grid(qr_img, modules_total):
    size = qr_img.size[0]
    module_px = size / modules_total
    grid = np.zeros((modules_total, modules_total), dtype=np.float32)
    for my in range(modules_total):
        for mx in range(modules_total):
            x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
            grid[my, mx] = 1.0 if module_is_dark(qr_img, x1, y1, x2, y2) else 0.0
    return grid


def logo_grid_score(qr_img, logo_path, qr, border):
    """QR maske adayini logo siluetiyle ne kadar uyumlu olduguna gore puanlar."""
    size = qr_img.size[0]
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    maps = logo_detail_maps(logo_path, size)
    dark_grid = qr_dark_grid(qr_img, modules_total)
    score = 0.0
    for my in range(modules_total):
        for mx in range(modules_total):
            if is_finder_module(mx, my, modules_total, border):
                continue
            x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
            core = cell_mean(maps["core"], x1, y1, x2, y2)
            edge = cell_mean(maps["edge"], x1, y1, x2, y2)
            halo = cell_mean(maps["halo"], x1, y1, x2, y2)
            dark = dark_grid[my, mx]
            score += dark * (core * 1.4 + edge * 2.1)
            score -= dark * halo * 0.55
    return score


def make_logo_aware_qr_png(data, path, size, error="h", border=4, logo_path=None, trials=8):
    """Ayni QR verisi icin maske adaylarini deneyip logo ile en uyumlu matrisi kaydeder."""
    if not logo_path or not globals().get("logo_aware_qr_mask_optimize", True):
        return make_qr_png(data, path, size, error=error, border=border)

    best = None
    max_trials = max(1, min(8, int(trials)))
    for mask_id in range(max_trials):
        try:
            qr = segno.make(data, error=error, boost_error=True, mask=mask_id)
        except TypeError:
            return make_qr_png(data, path, size, error=error, border=border)
        tmp_path = path.replace(".png", f"_mask_{mask_id}.png")
        modules_with_border = qr.symbol_size(scale=1, border=border)[0]
        scale = max(1, size // modules_with_border)
        qr.save(tmp_path, kind="png", scale=scale, border=border, dark="black", light="white")
        img = Image.open(tmp_path).convert("L")
        if img.size != (size, size):
            img = img.resize((size, size), Image.Resampling.NEAREST)
            img.save(tmp_path)
        score = logo_grid_score(img, logo_path, qr, border)
        if best is None or score > best[0]:
            best = (score, qr, img, mask_id)
        try:
            os.remove(tmp_path)
        except OSError:
            pass

    if best is None:
        return make_qr_png(data, path, size, error=error, border=border)
    _, qr, img, mask_id = best
    img.save(path)
    print(f"Logo-aware QR maskesi secildi: mask={mask_id}, score={best[0]:.3f}")
    return qr, img


def render_halftone_logo_qr(
    qr,
    qr_img,
    size,
    border,
    logo_path,
    mark_shape="plus",
    density=0.92,
    logo_strength=0.92,
    background_dots=True,
    designer_finders=True,
    logo_forward=True,
    light_logo_alpha=0.82,
    logo_cell_threshold=0.08,
    logo_aware=True,
    outline_boost=0.55,
    edge_weight=0.85,
    negative_cutout=0.45,
    safe_background_density=0.58,
    logo_bitmap_underlay=False,
    logo_underlay_alpha=0.0,
):
    if qr is None or not logo_path:
        return None
    dark_rgb = hex_to_rgb(dark_hex)
    brand_rgb = hex_to_rgb(brand_hex)
    light_rgb = hex_to_rgb(light_hex)
    maps = logo_detail_maps(logo_path, size)
    logo_layer = maps["layer"]
    logo_mask = maps["mask"]
    canvas = Image.new("RGBA", (size, size), light_rgb + (255,))
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    finder_origins = finder_module_origins(modules_total, border)
    structure = qr_structural_module_sets(modules_total, border)
    finder_set = structure["finder_core"]
    finder_guard_set = structure["finder_guard"]
    protected_set = structure["protected"]

    if logo_bitmap_underlay and logo_forward:
        underlay = logo_layer.copy()
        alpha = np.array(underlay.getchannel("A"), dtype=np.float32)
        logo_alpha = np.zeros_like(alpha, dtype=np.float32)
        visible_logo = np.logical_or(logo_mask, maps["edge"])
        logo_alpha[visible_logo] = alpha[visible_logo] * float(np.clip(logo_underlay_alpha, 0.0, 0.85))
        for mx, my in finder_guard_set:
            x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
            logo_alpha[y1:y2, x1:x2] = 0
        underlay.putalpha(Image.fromarray(np.clip(logo_alpha, 0, 255).astype(np.uint8), mode="L"))
        canvas.alpha_composite(underlay)

    draw = ImageDraw.Draw(canvas)

    for my in range(modules_total):
        for mx in range(modules_total):
            x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
            if (mx, my) in protected_set:
                continue
            dark = module_is_dark(qr_img, x1, y1, x2, y2)
            logo_coverage = cell_mean(logo_mask, x1, y1, x2, y2)
            core_coverage = cell_mean(maps["core"], x1, y1, x2, y2)
            edge_coverage = cell_mean(maps["edge"], x1, y1, x2, y2)
            halo_coverage = cell_mean(maps["halo"], x1, y1, x2, y2)
            light_detail = cell_mean(maps["light_detail"], x1, y1, x2, y2)
            inside_logo = logo_coverage > logo_cell_threshold or edge_coverage > logo_cell_threshold * 0.35
            near_finder = (mx, my) in finder_guard_set
            in_quiet_zone = mx < border or my < border or mx >= modules_total - border or my >= modules_total - border

            # Sessiz bolge temiz kalmali. Ilk ciktidaki dis cerceve arti desenleri
            # telefon okuyuculari ve gorsel algiyi ayni anda zayiflatiyordu.
            if in_quiet_zone:
                continue

            if light_detail > 0.42 and inside_logo and not dark and not near_finder:
                continue

            if not dark and not (background_dots and (inside_logo or edge_coverage > 0.02)):
                continue

            fallback = brand_rgb + (255,) if inside_logo else dark_rgb + (255,)
            sampled = logo_sample_color(logo_layer, x1, y1, x2, y2, fallback)
            edge_factor = min(1.0, edge_coverage * (1.0 + edge_weight))

            if logo_aware and inside_logo and logo_forward and not near_finder:
                color_mix = max(0.0, 1.0 - logo_strength)
                if edge_factor > 0.06:
                    fill_rgb = blend_rgb(sampled[:3], dark_rgb, max(0.10, color_mix * 0.45))
                    fill = fill_rgb + (255,)
                    local_density = min(1.0, density + outline_boost * 0.20 + edge_factor * 0.10)
                elif dark:
                    fill = blend_rgb(sampled[:3], dark_rgb, color_mix) + (255,)
                    local_density = min(1.0, density + core_coverage * 0.05)
                else:
                    # QR'in beyaz hucrelerinde de logoyu ayni nokta/plus diliyle tasiriz.
                    # Boylece logo arkaya silik basilmaz; modullerin icinde okunur.
                    local_logo_power = min(1.0, logo_coverage + edge_factor * 0.65 + core_coverage * 0.25)
                    alpha = int(round(255 * min(0.98, max(light_logo_alpha, 0.76 + local_logo_power * 0.20))))
                    light_mix = max(0.0, 1.0 - light_logo_alpha) * max(0.18, 1.0 - local_logo_power * 0.70)
                    fill_rgb = blend_rgb(sampled[:3], light_rgb, light_mix)
                    fill = fill_rgb + (alpha,)
                    local_density = min(1.0, max(0.62, density * (0.76 + 0.24 * local_logo_power) + edge_factor * 0.08))
                if light_detail > 0.25 and negative_cutout > 0:
                    # Logodaki beyaz/negatif detaylari gri lekeye cevirmek yerine
                    # modulu kucultur ve acik renge cekeriz; logo daha okunur kalir.
                    fill = blend_rgb(fill[:3], light_rgb, negative_cutout * 0.90) + (fill[3],)
                    local_density = max(0.22, local_density * (1.0 - negative_cutout * 0.48))
            elif dark:
                if inside_logo:
                    fill = blend_rgb(sampled[:3], dark_rgb, 1.0 - logo_strength) + (255,)
                    local_density = min(1.0, density + edge_factor * 0.06)
                else:
                    fill = dark_rgb + (255,)
                    local_density = max(0.50, safe_background_density)
            else:
                fill = blend_rgb(sampled[:3], light_rgb, 0.46) + (155,)
                local_density = max(0.26, density * 0.42)

            if halo_coverage > 0.08 and not inside_logo and not dark:
                fill = blend_rgb(brand_rgb, light_rgb, 0.55) + (150,)
                local_density = max(0.30, density * 0.42)

            draw_halftone_mark(draw, (x1, y1, x2, y2), mark_shape, fill, local_density)

    if logo_aware and outline_boost > 0:
        edge_alpha = np.clip(maps["edge"].astype(np.uint8) * int(120 * outline_boost), 0, 160).astype(np.uint8)
        edge_img = Image.fromarray(edge_alpha, mode="L").filter(ImageFilter.GaussianBlur(radius=max(0.4, size / 2200)))
        edge_layer = Image.new("RGBA", (size, size), dark_rgb + (0,))
        edge_layer.putalpha(edge_img)
        canvas.alpha_composite(edge_layer)

    non_finder_protected = protected_set - finder_guard_set
    draw_raw_qr_modules(draw, qr_img, non_finder_protected, module_px, "square", dark_rgb + (255,), light_rgb + (255,))

    if designer_finders:
        accents = [brand_rgb + (255,), blend_rgb(brand_rgb, dark_rgb, 0.25) + (255,), brand_rgb + (255,)]
        for (fx, fy), accent in zip(finder_origins, accents):
            x1 = int(round((fx - 1) * module_px)); y1 = int(round((fy - 1) * module_px))
            x2 = int(round((fx + 8) * module_px)); y2 = int(round((fy + 8) * module_px))
            draw_designer_finder(draw, (x1, y1, x2, y2), accent, light_rgb + (255,), dark_rgb + (255,))
    else:
        draw_raw_qr_modules(draw, qr_img, finder_set, module_px, "square", dark_rgb + (255,))
    return canvas

def render_best_readable_halftone_qr(qr, qr_img, size, border, logo_path, expected_data=None, out_dir="qr_fusion_outputs", prefix="halftone_probe"):
    """Birden fazla halftone profili dener; sadece decoder testinden geceni teslim/final icin dondurur."""
    os.makedirs(out_dir, exist_ok=True)
    base = dict(
        mark_shape=globals().get("halftone_mark_shape", "plus"),
        density=float(globals().get("halftone_density", 0.90)),
        logo_strength=float(globals().get("halftone_logo_strength", 0.88)),
        light_logo_alpha=float(globals().get("halftone_light_logo_alpha", 0.78)),
        logo_cell_threshold=float(globals().get("halftone_logo_cell_threshold", 0.07)),
        outline_boost=float(globals().get("halftone_outline_boost", 0.52)),
        edge_weight=float(globals().get("halftone_edge_weight", 0.80)),
        negative_cutout=float(globals().get("halftone_negative_cutout", 0.18)),
        safe_background_density=float(globals().get("halftone_safe_background_density", 0.74)),
        logo_underlay_alpha=0.0,
    )
    profiles = [
        dict(name="logo_forward_plus", **{**base, "density": 0.94, "logo_strength": 0.96, "light_logo_alpha": 0.94, "logo_cell_threshold": 0.045, "outline_boost": 0.64, "edge_weight": 1.20, "negative_cutout": 0.34, "safe_background_density": 0.78, "logo_underlay_alpha": 0.0}),
        dict(name="poster_style_plus", **{**base, "density": 0.90, "logo_strength": 0.92, "light_logo_alpha": 0.88, "logo_cell_threshold": 0.055, "outline_boost": 0.52, "edge_weight": 1.10, "negative_cutout": 0.38, "safe_background_density": 0.84, "logo_underlay_alpha": 0.0}),
        dict(name="balanced", **base),
        dict(name="scan_safe_plus", **{**base, "density": 0.84, "logo_strength": 0.82, "light_logo_alpha": 0.72, "logo_cell_threshold": 0.085, "outline_boost": 0.36, "negative_cutout": 0.10, "safe_background_density": 0.88, "logo_underlay_alpha": 0.0}),
        dict(name="dot_soft", **{**base, "mark_shape": "dot", "density": 0.86, "logo_strength": 0.80, "light_logo_alpha": 0.70, "logo_cell_threshold": 0.095, "outline_boost": 0.28, "negative_cutout": 0.08, "safe_background_density": 0.90, "logo_underlay_alpha": 0.0}),
        dict(name="square_high_contrast", **{**base, "mark_shape": "square", "density": 0.88, "logo_strength": 0.78, "light_logo_alpha": 0.64, "logo_cell_threshold": 0.11, "outline_boost": 0.20, "negative_cutout": 0.00, "safe_background_density": 0.92, "logo_underlay_alpha": 0.0}),
    ]
    rows = []
    for idx, profile in enumerate(profiles, start=1):
        params = {k: v for k, v in profile.items() if k != "name"}
        img = render_halftone_logo_qr(
            qr=qr,
            qr_img=qr_img,
            size=size,
            border=border,
            logo_path=logo_path,
            background_dots=globals().get("halftone_background_dots", True),
            designer_finders=globals().get("halftone_designer_finders", True),
            logo_forward=globals().get("halftone_logo_forward", True),
            logo_aware=globals().get("halftone_logo_aware_mode", True),
            **params,
        )
        path = os.path.join(out_dir, f"{prefix}_{idx:02d}_{profile['name']}.png")
        save_rgb_or_rgba(img, path)
        result = test_qr_all(path, expected_data=expected_data)
        quality = commercial_quality_grade(path, result=result, logo_path=logo_path, qr_img=qr_img, qr=qr, border=border)
        rows.append({"path": path, "image": img, "result": result, "quality": quality, "profile": profile})
        print(path, "OK" if result["ok"] else "NO", result["texts"], "score", quality["score"])
    readable = [row for row in rows if row["result"].get("ok")]
    if not readable:
        return None, rows
    readable.sort(key=lambda row: (row["quality"]["score"], row["result"].get("decoder_pass_count", 0)), reverse=True)
    return readable[0], rows

def visual_quality_metrics(path, logo_path=None, qr_img=None, qr=None, border=None):
    img = Image.open(path).convert("RGB")
    gray = np.array(img.convert("L"), dtype=np.float32)
    small = img.resize((256, 256), Image.Resampling.LANCZOS)
    small_gray = np.array(small.convert("L"), dtype=np.float32)
    metrics = {
        "contrast_std": round(float(gray.std()), 2),
        "small_preview_contrast": round(float(small_gray.std()), 2),
    }
    if qr_img is not None and qr is not None and border is not None:
        modules_total = qr.symbol_size(scale=1, border=border)[0]
        module_px = img.size[0] / modules_total
        structure = qr_structural_module_sets(modules_total, border)
        protected_scores = []
        for mx, my in structure["protected"]:
            x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
            patch = gray[y1:y2, x1:x2]
            if patch.size == 0:
                continue
            mean = float(patch.mean())
            expected_dark = module_is_dark(qr_img, x1, y1, x2, y2)
            protected_scores.append((255.0 - mean) / 255.0 if expected_dark else mean / 255.0)
        metrics["protected_structure_score"] = round(float(np.mean(protected_scores)) if protected_scores else 0.0, 3)
        q = int(round(border * module_px))
        if q > 0:
            quiet = np.concatenate([gray[:q, :].ravel(), gray[-q:, :].ravel(), gray[:, :q].ravel(), gray[:, -q:].ravel()])
            metrics["quiet_zone_cleanliness"] = round(float((quiet > 238).mean()), 3)
    if logo_path:
        try:
            _, mask = make_logo_mask(logo_path, img.size[0])
            metrics["logo_silhouette_coverage"] = round(float(mask.mean()), 3)
            logo_pixels = gray[mask]
            bg_pixels = gray[~mask]
            if logo_pixels.size and bg_pixels.size:
                metrics["logo_visibility_score"] = round(float((bg_pixels.mean() - logo_pixels.mean()) / 255.0), 3)
                metrics["logo_dark_pixel_ratio"] = round(float((logo_pixels < 142).mean()), 3)
        except Exception:
            metrics["logo_silhouette_coverage"] = None
    return metrics

def commercial_quality_grade(path, result=None, logo_path=None, qr_img=None, qr=None, border=None):
    visual = visual_quality_metrics(path, logo_path=logo_path, qr_img=qr_img, qr=qr, border=border)
    decoder_passes = 0 if result is None else int(result.get("decoder_pass_count", 0) or 0)
    required_passes = int(globals().get("min_decoder_passes", 1) or 1)
    contrast = float(visual.get("contrast_std", 0) or 0)
    preview_contrast = float(visual.get("small_preview_contrast", 0) or 0)
    logo_coverage = visual.get("logo_silhouette_coverage")
    logo_visibility = float(visual.get("logo_visibility_score", 0) or 0)
    logo_dark_ratio = float(visual.get("logo_dark_pixel_ratio", 0) or 0)

    score = 0
    score += min(45, decoder_passes * 15)
    score += 20 if result and result.get("ok") else 0
    score += max(0, min(20, (contrast - 28) / 42 * 20))
    score += max(0, min(10, (preview_contrast - 22) / 34 * 10))
    if logo_coverage is not None:
        if 0.05 <= logo_coverage <= 0.62:
            score += 5
        elif logo_coverage > 0.75:
            score -= 6
    score += max(0, min(10, logo_visibility / 0.36 * 10))
    score += max(0, min(5, logo_dark_ratio / 0.42 * 5))
    protected_score = visual.get("protected_structure_score")
    quiet_cleanliness = visual.get("quiet_zone_cleanliness")
    if protected_score is not None:
        score += max(0, min(10, (float(protected_score) - 0.72) / 0.24 * 10))
    if quiet_cleanliness is not None:
        score += max(0, min(8, (float(quiet_cleanliness) - 0.92) / 0.08 * 8))
    score = round(float(max(0, min(100, score))), 1)

    if score >= 86 and decoder_passes >= required_passes:
        grade = "premium_ready"
    elif score >= 72 and decoder_passes >= required_passes:
        grade = "delivery_ready"
    elif decoder_passes >= required_passes:
        grade = "readable_needs_visual_review"
    else:
        grade = "visual_only_not_delivery_safe"

    notes = []
    if decoder_passes < required_passes:
        notes.append("Decoder pass sayisi ticari kabul icin dusuk; final olarak kullanma.")
    if contrast < 35:
        notes.append("Kontrast dusuk; baski veya dusuk isikta okuma riski artabilir.")
    if preview_contrast < 28:
        notes.append("Kucuk onizlemede modul ayrimi zayif gorunuyor.")
    if logo_coverage is not None and logo_coverage > 0.72:
        notes.append("Logo alan kaplamasi yuksek; daha kisa URL veya daha sade logo onerilir.")
    if logo_path and logo_visibility < 0.12:
        notes.append("Logo nokta/plus katmaninda zayif; logo_forward_plus veya poster_style_plus varyantini manuel incele.")
    if protected_score is not None and float(protected_score) < 0.78:
        notes.append("QR yapisal modulleri fazla stilize; finder/timing/alignment korumasini artir.")
    if quiet_cleanliness is not None and float(quiet_cleanliness) < 0.96:
        notes.append("Sessiz bolge yeterince temiz degil; dis dekorlari azalt.")
    if not notes:
        notes.append("Okunabilirlik ve gorsel kontrast satis teslimi icin dengeli gorunuyor.")

    return {
        "score": score,
        "grade": grade,
        "decoder_passes": decoder_passes,
        "required_passes": required_passes,
        "visual": visual,
        "notes": notes,
    }

SHAPE_DELIVERY_GUIDE = {
    "square": dict(scan_safety="highest", best_for="print, label, black-white clean", risk="least artistic but most robust"),
    "rounded": dict(scan_safety="high", best_for="business, brand QR, general premium delivery", risk="safe default"),
    "dot": dict(scan_safety="medium", best_for="event, social, playful brand", risk="small print can soften modules"),
    "diamond": dict(scan_safety="medium", best_for="editorial or luxury accent variants", risk="corners can reduce decoder tolerance"),
    "hexagon": dict(scan_safety="medium", best_for="tech/geometric brand variants", risk="needs strong contrast"),
    "vertical_line": dict(scan_safety="low", best_for="visual variant only", risk="module centers may look connected"),
    "horizontal_line": dict(scan_safety="low", best_for="visual variant only", risk="module centers may look connected"),
    "soft_blob": dict(scan_safety="medium", best_for="colorful/social preview", risk="avoid tiny print sizes"),
    "plus": dict(scan_safety="high", best_for="halftone logo-forward premium final", risk="best with high resolution export"),
    "star": dict(scan_safety="low", best_for="accent/portfolio variant", risk="decorative shape, test before delivery"),
}

def variant_shape_name(row):
    variant = row.get("variant") or {}
    return row.get("logo_mode") or variant.get("module_shape") or variant.get("mark_shape") or globals().get("module_shape", "rounded")

def build_shape_quality_diagnostics(rows, logo_path=None):
    diagnostics = []
    active_qr_img = globals().get("qr_img")
    active_qr = globals().get("qr")
    active_border = globals().get("quiet_zone", 4)
    for row in rows:
        path = row.get("path")
        if not path or not os.path.exists(path):
            continue
        result = row.get("result") or {}
        shape = variant_shape_name(row)
        quality = commercial_quality_grade(path, result=result, logo_path=logo_path, qr_img=active_qr_img, qr=active_qr, border=active_border)
        guide = SHAPE_DELIVERY_GUIDE.get(shape, {})
        diagnostics.append({
            "path": path,
            "shape": shape,
            "ok": bool(result.get("ok")),
            "decoder_pass_count": result.get("decoder_pass_count"),
            "passed_engines": result.get("passed_engines"),
            "score": quality["score"],
            "grade": quality["grade"],
            "scan_safety": guide.get("scan_safety", "unknown"),
            "best_for": guide.get("best_for", "custom delivery"),
            "risk": guide.get("risk", "manual phone scan recommended"),
            "notes": quality["notes"],
        })
    diagnostics.sort(key=lambda item: (item["ok"], item["score"]), reverse=True)
    return diagnostics

def write_shape_quality_report(path, diagnostics):
    with open(path, "w", encoding="utf-8") as f:
        f.write("Shape-Based Quality Diagnosis\n")
        f.write("=============================\n\n")
        f.write("Use the top readable rows for client delivery; keep low-safety shapes as portfolio/visual extras unless phone-tested.\n\n")
        for idx, item in enumerate(diagnostics, start=1):
            f.write(f"{idx}. {item['path']}\n")
            f.write(f"  shape: {item['shape']}\n")
            f.write(f"  grade: {item['grade']} | score: {item['score']} | ok: {item['ok']}\n")
            f.write(f"  decoders: {item['decoder_pass_count']} | engines: {item['passed_engines']}\n")
            f.write(f"  scan_safety: {item['scan_safety']}\n")
            f.write(f"  best_for: {item['best_for']}\n")
            f.write(f"  risk: {item['risk']}\n")
            for note in item.get("notes", []):
                f.write(f"  note: {note}\n")
            f.write("\n")
    return path

def write_client_quality_summary(path, final_path, final_result, diagnostics):
    final_quality = commercial_quality_grade(
        final_path,
        result=final_result,
        logo_path=globals().get("prepared_logo_path") if globals().get("logo_source_mode") == "uploaded_artwork" else None,
    )
    best = diagnostics[0] if diagnostics else None
    readable = [item for item in diagnostics if item.get("ok")]
    with open(path, "w", encoding="utf-8") as f:
        f.write("Client Quality Summary\n")
        f.write("======================\n\n")
        f.write(f"Final file: {final_path}\n")
        f.write(f"Final grade: {final_quality['grade']}\n")
        f.write(f"Final score: {final_quality['score']} / 100\n")
        f.write(f"Decoder passes: {final_quality['decoder_passes']} / {final_quality['required_passes']}\n")
        f.write(f"Passed engines: {final_result.get('passed_engines')}\n")
        f.write(f"Output size: {globals().get('output_size')} px\n")
        f.write(f"Print setting: {globals().get('print_size_preset')} at {globals().get('print_dpi')} dpi\n")
        f.write(f"Readable commercial variants: {len(readable)} / {len(diagnostics)}\n\n")
        if best:
            f.write("Best scored variant:\n")
            f.write(f"- path: {best['path']}\n")
            f.write(f"- shape: {best['shape']}\n")
            f.write(f"- score: {best['score']}\n")
            f.write(f"- grade: {best['grade']}\n\n")
        f.write("Seller notes:\n")
        for note in final_quality.get("notes", []):
            f.write(f"- {note}\n")
        f.write("- Manual phone scan is still recommended before sending the order as complete.\n")
    return path


def is_finder_module(mx, my, modules_total, border):
    qr_size_no_border = modules_total - 2 * border
    zones = [
        (border, border),
        (border + qr_size_no_border - 7, border),
        (border, border + qr_size_no_border - 7),
    ]
    for zx, zy in zones:
        if zx <= mx < zx + 7 and zy <= my < zy + 7:
            return True
    return False

def finder_pixel_boxes(size, qr, border):
    if qr is None:
        return []
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    qr_size_no_border = modules_total - 2 * border
    zones = [
        (border, border),
        (border + qr_size_no_border - 7, border),
        (border, border + qr_size_no_border - 7),
    ]
    boxes = []
    for mx, my in zones:
        x1 = max(0, int(math.floor(mx * module_px)))
        y1 = max(0, int(math.floor(my * module_px)))
        x2 = min(size, int(math.ceil((mx + 7) * module_px)))
        y2 = min(size, int(math.ceil((my + 7) * module_px)))
        boxes.append((x1, y1, x2, y2))
    return boxes

def module_color(mx, my, modules_total, mode, dark_rgb, brand_rgb, grad_a, grad_b, artwork_arr=None):
    if mode == "image_based" and artwork_arr is not None:
        h, w = artwork_arr.shape[:2]
        px = min(w - 1, max(0, int(round((mx + 0.5) / modules_total * w))))
        py = min(h - 1, max(0, int(round((my + 0.5) / modules_total * h))))
        sampled = tuple(int(v) for v in artwork_arr[py, px, :3])
        return blend_rgb(sampled, dark_rgb, 0.45) + (255,)
    if mode == "brand_color":
        return brand_rgb + (255,)
    if mode == "gradient":
        t = (mx + my) / max(1, (modules_total - 1) * 2)
        return blend_rgb(grad_a, grad_b, t) + (255,)
    if mode == "two_color":
        t = mx / max(1, modules_total - 1)
        return blend_rgb(dark_rgb, brand_rgb, t) + (255,)
    return dark_rgb + (255,)

def render_styled_qr(qr, qr_img, size, border, module_shape="rounded", finder_style="rounded", color_mode="brand_color", background_style="white", transparent=False, artwork=None):
    dark_rgb = hex_to_rgb(dark_hex)
    brand_rgb = hex_to_rgb(brand_hex)
    grad_a = hex_to_rgb(gradient_start_hex)
    grad_b = hex_to_rgb(gradient_end_hex)
    light_rgb = hex_to_rgb(light_hex)

    canvas = make_background(size, background_style, light_rgb, brand_rgb, artwork=artwork, transparent=transparent)
    draw = ImageDraw.Draw(canvas)
    artwork_arr = np.array(artwork.convert("RGB").resize((size, size), Image.Resampling.LANCZOS)) if artwork is not None else None
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total

    for my in range(modules_total):
        for mx in range(modules_total):
            x1 = int(round(mx * module_px))
            y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px))
            y2 = int(round((my + 1) * module_px))
            if not module_is_dark(qr_img, x1, y1, x2, y2):
                continue
            shape = module_shape
            if is_finder_module(mx, my, modules_total, border):
                shape = {"classic": "square", "rounded": "rounded", "circle": "dot", "planet": "dot"}.get(finder_style, module_shape)
            fill = module_color(mx, my, modules_total, color_mode, dark_rgb, brand_rgb, grad_a, grad_b, artwork_arr=artwork_arr)
            draw_module(draw, (x1, y1, x2, y2), shape, fill)

    if finder_style == "planet":
        finder_boxes = finder_pixel_boxes(size, qr, border)
        for box in finder_boxes:
            x1, y1, x2, y2 = box
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            r_outer = (x2 - x1) * 0.43
            r_inner = (x2 - x1) * 0.20
            accent = brand_rgb + (255,)
            dark = dark_rgb + (255,)
            light = light_rgb + (255,)
            draw.ellipse((cx - r_inner, cy - r_inner, cx + r_inner, cy + r_inner), fill=dark)
            draw.ellipse((cx - r_outer, cy - r_outer, cx + r_outer, cy + r_outer), outline=dark, width=max(2, int((x2 - x1) * 0.055)))
            dot_r = max(2, int((x2 - x1) * 0.055))
            for ang in (0, math.pi / 2, math.pi, math.pi * 1.5):
                dx = math.cos(ang) * r_outer
                dy = math.sin(ang) * r_outer
                draw.ellipse((cx + dx - dot_r, cy + dy - dot_r, cx + dx + dot_r, cy + dy + dot_r), fill=accent)

    return canvas

def apply_logo_treatment(base_img, logo_path, mode="center_badge", size_ratio=0.16, badge_shape="rounded_square", badge_opacity=0.96, badge_padding=0.28, visual_strength=0.90):
    if mode == "none" or not logo_path:
        return base_img.convert("RGBA")

    base = base_img.convert("RGBA")
    size = base.size[0]
    logo_box = max(1, int(round(size * size_ratio)))
    visual_strength = float(np.clip(visual_strength, 0.20, 1.00))
    logo_raw = Image.open(logo_path).convert("RGBA")

    # Rozet/kose modlarinda logo net kalmali; sadece arka plan modlarinda yumusatiladi.
    if mode in ("background_logo", "qr_over_logo_soft"):
        logo = soften_logo_for_qr(logo_raw, visual_strength=visual_strength)
    else:
        rgb = ImageEnhance.Contrast(logo_raw.convert("RGB")).enhance(0.95 + 0.35 * visual_strength)
        rgb = ImageEnhance.Color(rgb).enhance(0.95 + 0.25 * visual_strength)
        logo = rgb.convert("RGBA")
        logo.putalpha(logo_raw.getchannel("A"))

    logo = ImageOps.contain(logo, (logo_box, logo_box), Image.Resampling.LANCZOS)

    if mode == "qr_over_logo_soft":
        bg_logo = ImageOps.contain(logo, (int(size * 0.86), int(size * 0.86)), Image.Resampling.LANCZOS)
        alpha_factor = 0.18 + 0.24 * visual_strength
        alpha = bg_logo.getchannel("A").point(lambda p: int(p * alpha_factor))
        bg_logo.putalpha(alpha)
        layer = Image.new("RGBA", base.size, (255, 255, 255, 0))
        layer.paste(bg_logo, ((size - bg_logo.size[0]) // 2, (size - bg_logo.size[1]) // 2), bg_logo)
        return Image.alpha_composite(layer, base)

    if mode == "background_logo":
        bg_logo = ImageOps.contain(logo, (int(size * 0.72), int(size * 0.72)), Image.Resampling.LANCZOS)
        alpha_factor = 0.26 + 0.34 * visual_strength
        alpha = bg_logo.getchannel("A").point(lambda p: int(p * alpha_factor))
        bg_logo.putalpha(alpha)
        layer = Image.new("RGBA", base.size, (255, 255, 255, 0))
        layer.paste(bg_logo, ((size - bg_logo.size[0]) // 2, (size - bg_logo.size[1]) // 2), bg_logo)
        return Image.alpha_composite(layer, base)

    badge_box = int(round(logo_box * (1.0 + badge_padding * 2.0)))
    layer = Image.new("RGBA", base.size, (255, 255, 255, 0))
    if mode == "corner_small":
        margin = int(size * 0.045)
        bx = size - badge_box - margin
        by = size - badge_box - margin
    else:
        bx = (size - badge_box) // 2
        by = (size - badge_box) // 2

    if mode in ("center_badge", "corner_small", "logo_under_qr"):
        badge = Image.new("RGBA", (badge_box, badge_box), (255, 255, 255, 0))
        bdraw = ImageDraw.Draw(badge)
        fill = (255, 255, 255, int(255 * badge_opacity))
        if badge_shape == "circle":
            bdraw.ellipse((0, 0, badge_box - 1, badge_box - 1), fill=fill)
        elif badge_shape == "square":
            bdraw.rectangle((0, 0, badge_box - 1, badge_box - 1), fill=fill)
        else:
            radius = max(1, int(round(badge_box * 0.18)))
            bdraw.rounded_rectangle((0, 0, badge_box - 1, badge_box - 1), radius=radius, fill=fill)
        layer.paste(badge, (bx, by), badge)
    elif mode == "center_transparent":
        clear = Image.new("RGBA", (badge_box, badge_box), (255, 255, 255, int(255 * 0.45)))
        layer.paste(clear, (bx, by), clear)

    if mode == "corner_small":
        lx = bx + (badge_box - logo.size[0]) // 2
        ly = by + (badge_box - logo.size[1]) // 2
    else:
        lx = (size - logo.size[0]) // 2
        ly = (size - logo.size[1]) // 2
    layer.paste(logo, (lx, ly), logo)
    return Image.alpha_composite(base, layer)


def soften_logo_for_qr(logo, visual_strength=0.58):
    """Logoyu QR ustunde daha az baskin yapar; alfa ve kontrasti dusurur."""
    visual_strength = float(np.clip(visual_strength, 0.20, 1.00))
    logo = logo.convert("RGBA")
    rgb = logo.convert("RGB")
    rgb = ImageEnhance.Contrast(rgb).enhance(0.65 + 0.35 * visual_strength)
    rgb = ImageEnhance.Color(rgb).enhance(0.55 + 0.45 * visual_strength)
    rgb = Image.blend(Image.new("RGB", logo.size, "white"), rgb, visual_strength)
    softened = rgb.convert("RGBA")
    alpha = logo.getchannel("A").point(lambda p: int(p * (0.50 + 0.50 * visual_strength)))
    softened.putalpha(alpha)
    return softened

def logo_clearance_ratio(qr, border):
    """QR yogunlastikca merkezi kapatma alanini kucultur."""
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    data_modules = max(1, modules_total - 2 * border)
    if data_modules <= 29:
        return 0.22
    if data_modules <= 37:
        return 0.18
    return 0.15

def apply_logo_treatment_readable(
    base_img,
    logo_path,
    qr=None,
    border=4,
    expected_data=None,
    mode="center_badge",
    size_ratio=0.16,
    badge_shape="rounded_square",
    badge_opacity=0.96,
    badge_padding=0.28,
    visual_strength=0.90,
    auto_guard=True,
    test_path="qr_fusion_outputs/_logo_readability_probe.png",
):
    if mode == "none" or not logo_path:
        return base_img.convert("RGBA"), {"ok": True, "size_ratio": 0.0, "texts": []}

    capped_ratio = float(size_ratio)
    if qr is not None:
        capped_ratio = min(capped_ratio, logo_clearance_ratio(qr, border))

    ratios = [capped_ratio]
    if auto_guard:
        ratios = sorted(set(round(r, 3) for r in [capped_ratio, capped_ratio * 0.88, capped_ratio * 0.76, capped_ratio * 0.64, capped_ratio * 0.52]), reverse=True)

    best_img = None
    best_result = None
    os.makedirs(os.path.dirname(test_path) or ".", exist_ok=True)

    for ratio in ratios:
        candidate = apply_logo_treatment(
            base_img,
            logo_path,
            mode=mode,
            size_ratio=ratio,
            badge_shape=badge_shape,
            badge_opacity=badge_opacity,
            badge_padding=badge_padding,
            visual_strength=visual_strength,
        )
        save_rgb_or_rgba(candidate, test_path)
        result = test_qr_all(test_path, expected_data=expected_data)
        result["size_ratio"] = ratio
        if best_img is None:
            best_img, best_result = candidate, result
        if result["ok"]:
            return candidate, result

    return best_img, best_result

def save_rgb_or_rgba(img, path):
    if path.lower().endswith((".jpg", ".jpeg")):
        img.convert("RGB").save(path, quality=95)
    else:
        img.save(path)
    return path

def create_social_preview(img, path, canvas_size=1080):
    canvas = Image.new("RGB", (canvas_size, canvas_size), hex_to_rgb(light_hex))
    qr_side = int(canvas_size * 0.82)
    qr_preview = ImageOps.contain(img.convert("RGBA"), (qr_side, qr_side), Image.Resampling.LANCZOS)
    x = (canvas_size - qr_preview.size[0]) // 2
    y = (canvas_size - qr_preview.size[1]) // 2
    canvas.paste(qr_preview, (x, y), qr_preview)
    canvas.save(path, quality=95)
    return path

def add_scan_me_frame(img, path, text=None, style="rounded_brand"):
    base = img.convert("RGBA")
    size = base.size[0]
    brand = hex_to_rgb(brand_hex)
    dark = hex_to_rgb(dark_hex)
    light = hex_to_rgb(light_hex)
    pad = max(80, int(size * 0.08))
    label_h = max(160, int(size * 0.13))
    canvas = Image.new("RGB", (size + pad * 2, size + pad * 2 + label_h), light)
    draw = ImageDraw.Draw(canvas)
    qr_x = pad
    qr_y = pad
    if style == "bold_label":
        draw.rectangle((0, canvas.size[1] - label_h, canvas.size[0], canvas.size[1]), fill=brand)
        label_fill = light
    elif style == "minimal":
        draw.rectangle((pad // 2, pad // 2, canvas.size[0] - pad // 2, canvas.size[1] - pad // 2), outline=dark, width=max(3, size // 220))
        label_fill = dark
    else:
        radius = max(24, size // 45)
        draw.rounded_rectangle((pad // 2, pad // 2, canvas.size[0] - pad // 2, canvas.size[1] - pad // 2), radius=radius, outline=brand, width=max(6, size // 140))
        label_fill = brand
    canvas.paste(base, (qr_x, qr_y), base)
    label = (text or globals().get("scan_me_text", "SCAN ME") or "SCAN ME").strip()
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", max(42, int(label_h * 0.38)))
    except Exception:
        font = None
    bbox = draw.textbbox((0, 0), label, font=font)
    tx = (canvas.size[0] - (bbox[2] - bbox[0])) // 2
    ty = canvas.size[1] - label_h + (label_h - (bbox[3] - bbox[1])) // 2 - 4
    draw.text((tx, ty), label, fill=label_fill, font=font)
    canvas.save(path, quality=95, dpi=(print_dpi, print_dpi))
    return path

def print_dimensions_px():
    preset = globals().get("print_size_preset", "2x2_inch")
    if preset == "3x3_inch":
        width, height = 3.0, 3.0
    elif preset == "4x4_inch":
        width, height = 4.0, 4.0
    elif preset == "custom":
        width = float(globals().get("custom_print_width_in", 2.0) or 2.0)
        height = float(globals().get("custom_print_height_in", 2.0) or 2.0)
    else:
        width, height = 2.0, 2.0
    dpi = int(globals().get("print_dpi", 300) or 300)
    return max(300, int(round(width * dpi))), max(300, int(round(height * dpi))), dpi, width, height

def write_test_report(report_path, rows):
    from datetime import datetime
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("QR Fusion Commercial Test Report\n")
        f.write("================================\n")
        f.write(f"Test date: {datetime.now().isoformat(timespec='seconds')}\n")
        f.write(f"Service tier: {globals().get('service_tier', '')}\n")
        f.write(f"Portfolio example: {globals().get('portfolio_example', '')}\n")
        f.write(f"Usage place: {globals().get('usage_place', '')}\n")
        f.write(f"QR content type: {globals().get('qr_content_type', '')}\n")
        f.write(f"Static/dynamic note: {globals().get('qr_static_or_dynamic', '')}\n")
        f.write(f"QR data: {qr_data}\n")
        f.write(f"Expected decoded data: {globals().get('expected_data', qr_data)}\n")
        f.write(f"Required decoder passes: {globals().get('min_decoder_passes', 1)}\n")
        f.write(f"Error level: {error_level}\n")
        f.write(f"Quiet zone: {quiet_zone}\n")
        f.write(f"Module shape: {module_shape}\n")
        f.write(f"Finder style: {finder_style}\n")
        f.write(f"Color mode: {color_mode}\n")
        f.write(f"Logo mode: {logo_mode}\n")
        f.write("Vector note: artistic/logo-fused outputs are high-resolution raster files; clean base QR SVG is separate.\n")
        f.write("Print note: test one sample print before mass production because material, ink and lighting can affect scanning.\n\n")
        for row in rows:
            result = row['result']
            f.write(f"{row['path']}\n")
            f.write(f"  ok: {result.get('ok')}\n")
            f.write(f"  decoder_pass_count: {result.get('decoder_pass_count')}\n")
            f.write(f"  passed_engines: {result.get('passed_engines')}\n")
            f.write(f"  decoded_texts: {result.get('texts')}\n")
            f.write(f"  zbar: {result.get('zbar')}\n")
            f.write(f"  opencv: {result.get('opencv')}\n")
            f.write(f"  zxing: {result.get('zxing')}\n")
            if 'visual' in row:
                f.write(f"  visual: {row['visual']}\n")
            if 'quality' in row:
                f.write(f"  quality: {row['quality']}\n")
            if 'variant' in row:
                f.write(f"  variant: {row['variant']}\n")
            if 'logo_mode' in row:
                f.write(f"  logo_mode: {row['logo_mode']}\n")
            f.write("\n")
    return report_path


def write_customer_delivery_brief(path, final_result, delivery_files):
    values = {
        "client_name": globals().get("client_name", ""),
        "order_reference": globals().get("order_reference", ""),
        "service_tier": globals().get("service_tier", "premium"),
        "qr_data": globals().get("qr_data", ""),
        "expected_data": globals().get("expected_data", ""),
        "portfolio_example": globals().get("portfolio_example", "custom"),
        "style_preset": globals().get("style_preset", "custom"),
        "usage_place": globals().get("usage_place", ""),
        "qr_content_type": globals().get("qr_content_type", ""),
        "qr_static_or_dynamic": globals().get("qr_static_or_dynamic", ""),
        "print_size_preset": globals().get("print_size_preset", ""),
        "print_dpi": globals().get("print_dpi", ""),
        "scan_me_frame": globals().get("make_scan_me_frame", ""),
        "requested_style_note": globals().get("requested_style_note", ""),
        "revision_note": globals().get("revision_note", ""),
        "quality_mode": globals().get("quality_mode", ""),
        "delivery_quality_mode": globals().get("delivery_quality_mode", ""),
        "output_size": globals().get("output_size", ""),
        "error_level": globals().get("error_level", ""),
        "quiet_zone": globals().get("quiet_zone", ""),
        "min_decoder_passes": globals().get("min_decoder_passes", ""),
    }
    with open(path, "w", encoding="utf-8") as f:
        f.write("ArtQR Fusion Customer Delivery Brief\n")
        f.write("====================================\n\n")
        for key, value in values.items():
            f.write(f"{key}: {value}\n")
        f.write("\nFinal QR validation:\n")
        f.write(f"ok: {final_result.get('ok')}\n")
        f.write(f"decoder_passes: {final_result.get('decoder_pass_count')}\n")
        f.write(f"passed_engines: {final_result.get('passed_engines')}\n")
        f.write(f"decoded_texts: {final_result.get('texts')}\n")
        final_quality = commercial_quality_grade("qr_fusion_outputs/final_readable_art_qr.png", result=final_result, logo_path=globals().get("prepared_logo_path"))
        f.write(f"commercial_quality_grade: {final_quality.get('grade')}\n")
        f.write(f"commercial_quality_score: {final_quality.get('score')} / 100\n")
        f.write("\nDelivery notes:\n")
        f.write("- Artistic/logo-fused QR files are high-resolution raster outputs.\n")
        f.write("- Clean vector SVG/EPS, when included, is the base QR style and not the artistic halftone raster artwork.\n")
        f.write("- Static QR codes do not expire; dynamic editing requires the client to provide a dynamic destination/link service.\n")
        f.write("- For print jobs, test one physical sample before bulk printing.\n")
        f.write("\nDelivered files:\n")
        for item in delivery_files:
            f.write(f"- {item}\n")
    return path

def write_gig_readiness_checklist(path, final_result, delivery_files):
    required = {
        "final_readable_art_qr.png": os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"),
        "clean_qr.png": os.path.exists("qr_fusion_outputs/clean_qr.png"),
        "prepared_artwork.png": os.path.exists("qr_fusion_outputs/prepared_artwork.png"),
        "qr_test_report.txt": any(str(x).endswith("qr_test_report.txt") for x in delivery_files),
        "print_ready_file": any("print_ready" in str(x) for x in delivery_files),
        "customer_delivery_brief.txt": any(str(x).endswith("customer_delivery_brief.txt") for x in delivery_files),
    }
    commercial_ready = bool(final_result.get("ok")) and all(required.values())
    with open(path, "w", encoding="utf-8") as f:
        f.write("Fiverr / Upwork Gig Readiness Checklist\n")
        f.write("=======================================\n\n")
        f.write(f"Commercial ready: {commercial_ready}\n")
        f.write(f"Decoder requirement: {final_result.get('decoder_pass_count')} / {globals().get('min_decoder_passes', 1)}\n\n")
        for key, value in required.items():
            mark = "x" if value else " "
            f.write(f"[{mark}] {key}\n")
        f.write("\nBuyer must provide: target link/text, logo/artwork, preferred style, usage place, revision notes.\n")
        f.write("Seller should manually scan final PNG on a phone before delivery.\n")
    return path


def create_layout_canvas(qr_img, path, layout="sticker", title=""):
    base = qr_img.convert("RGBA")
    brand = hex_to_rgb(brand_hex)
    light = hex_to_rgb(light_hex)
    if layout == "business_card":
        canvas = Image.new("RGB", (1600, 1000), light)
        qr_side = 560
        qr_small = ImageOps.contain(base, (qr_side, qr_side), Image.Resampling.LANCZOS)
        canvas.paste(qr_small, (940, 220), qr_small)
        draw = ImageDraw.Draw(canvas)
        draw.rectangle((0, 0, 72, 1000), fill=brand)
        draw.text((150, 330), title or brand_text or "SCAN ME", fill=dark_hex)
    elif layout == "label":
        canvas = Image.new("RGB", (1400, 1800), light)
        qr_side = 1040
        qr_small = ImageOps.contain(base, (qr_side, qr_side), Image.Resampling.LANCZOS)
        canvas.paste(qr_small, ((1400 - qr_small.size[0]) // 2, 210), qr_small)
        ImageDraw.Draw(canvas).text((140, 1380), title or brand_text or qr_data, fill=dark_hex)
    else:
        canvas = Image.new("RGB", (1200, 1200), light)
        qr_side = 900
        qr_small = ImageOps.contain(base, (qr_side, qr_side), Image.Resampling.LANCZOS)
        canvas.paste(qr_small, ((1200 - qr_small.size[0]) // 2, 115), qr_small)
        ImageDraw.Draw(canvas).text((150, 1040), title or brand_text or "SCAN", fill=dark_hex)
    canvas.save(path, quality=95)
    return path

def save_segno_vector(qr, svg_path, eps_path=None):
    if qr is None:
        return []
    paths = []
    qr.save(svg_path, kind="svg", border=quiet_zone, dark=dark_hex, light=light_hex)
    paths.append(svg_path)
    if eps_path:
        qr.save(eps_path, kind="eps", border=quiet_zone, dark=dark_hex, light=light_hex)
        paths.append(eps_path)
    return paths





<!-- artqr-doc -->
## 8. Temel Dosyalari Uretme ve Kontrol Etme

Yuklenen gorsel hedef boyuta hazirlanir, temiz QR kod uretilir ve ikisi de ekranda gosterilir. Ardindan temiz QR kod test edilir; temiz QR okunmuyorsa sonraki gorsel birlestirme adimlarina gecmeden hata verilir.


In [ ]:
# Bu blok gercek uretimi baslatir: logo hazirlanir, logo-aware QR maskesi secilir, temiz QR test edilir ve ilk onizlemeler kaydedilir.
os.makedirs("qr_fusion_outputs", exist_ok=True)

pad_color = (pad_red, pad_green, pad_blue)
logo_quality = analyze_logo_quality(artwork_path)
print("Logo/artwork kalite analizi:", logo_quality)
for warning in logo_quality["warnings"]:
    print("UYARI:", warning)
prepared_logo_source_path = prepare_logo_source(artwork_path)
prepared_logo_path = prepare_fullscreen_logo_artwork(
    prepared_logo_source_path,
    out_path="qr_fusion_outputs/prepared_logo_fullscreen.png",
    size=output_size,
)
art_img = prepare_artwork(prepared_logo_path, output_size, fit_mode="contain_pad", pad_color=pad_color)

if upload_mode == "use_uploaded_qr_image":
    qr = None
    qr_img = load_uploaded_qr_image(uploaded_qr_path, output_size)
    qr_img.save("qr_fusion_outputs/clean_qr.png")
    qr_designator = "uploaded_qr_image"
else:
    qr, qr_img = make_logo_aware_qr_png(
        qr_data,
        "qr_fusion_outputs/clean_qr.png",
        output_size,
        error=error_level,
        border=quiet_zone,
        logo_path=prepared_logo_path,
        trials=logo_aware_mask_trials,
    )
    qr_designator = qr.designator

art_img.save("qr_fusion_outputs/prepared_artwork.png", quality=95)

print("QR version/designator:", qr_designator)
print("Artwork size:", art_img.size)
print("Tam ekran hazir logo:", prepared_logo_path)
display(Image.open(prepared_logo_path).convert("RGBA"))
display(art_img)
display(qr_img)

clean_test = test_qr_all("qr_fusion_outputs/clean_qr.png", expected_data=expected_data)
print("Temiz QR test:", clean_test)

if not clean_test["ok"]:
    raise ValueError("Temiz QR ticari kabul kuralindan gecmedi. Hazir QR modunda expected_qr_data/qr_data degerini, ya da min_decoder_passes ayarini kontrol et.")



<!-- artqr-doc -->
## 9. Gorsel-QR Adaylarini Uretme

Farkli karisim guclerinde birden fazla aday QR gorseli uretilir. Her aday otomatik olarak test edilir; okunabilen adaylar kaydedilir ve ayara gore ilk basarili sonuc final olarak secilebilir.


In [ ]:
# Bu blok farkli fusion guclerinde adaylar uretir, okuyucu testlerinden gecirir ve en iyi okunabilir sonucu secer.
candidate_paths = []
passed_candidates = []
scored_candidates = []

strength_values = np.arange(
    dark_strength_start,
    dark_strength_end + 0.0001,
    dark_strength_step,
)

for idx, strength in enumerate(strength_values, start=1):
    fused = luminance_fusion(
        art_img=art_img,
        qr_img=qr_img,
        qr=qr,
        border=quiet_zone,
        dark_strength=float(strength),
        light_ratio=light_strength_ratio,
        finder_boost=finder_boost,
        quiet_boost=quiet_zone_boost,
        module_contrast=module_contrast,
        center_bias=center_bias,
    )
    fused = polish_image(fused, contrast=contrast_after, sharpness=sharpness_after)

    path = f"qr_fusion_outputs/candidate_{idx:02d}_strength_{strength:.2f}.png"
    fused.save(path, quality=95)
    candidate_paths.append(path)

    result = test_qr_all(path, expected_data=expected_data)
    score = qr_quality_score(path, result, strength=float(strength))
    scored_candidates.append((score, path, float(strength), result))
    print(path, "OK" if result["ok"] else "NO", "score", round(score, 2), result["passed_engines"], result["texts"])

    if show_candidate_previews:
        display(fused.resize((384, 384), Image.Resampling.LANCZOS))

    if result["ok"]:
        passed_candidates.append((path, float(strength), result, score))
        if not save_all_candidates:
            break

print("\nEn iyi aday ozeti:")
for rank, (score, path, strength, result) in enumerate(sorted(scored_candidates, key=lambda item: item[0], reverse=True)[:6], start=1):
    print(f"{rank}. score={score:.2f} strength={strength:.2f} ok={result['ok']} decoders={result['passed_engines']} path={path}")

if not passed_candidates:
    print("Henuz okunabilir sonuc yok. dark_strength_end, finder_boost, module_contrast veya quiet_zone_boost artir.")
else:
    final_path = "qr_fusion_outputs/final_readable_art_qr.png"
    if prefer_best_score_not_first_pass:
        best_path, best_strength, best_result, best_score = max(passed_candidates, key=lambda item: item[3])
    else:
        best_path, best_strength, best_result, best_score = passed_candidates[0]
    final_img = Image.open(best_path).convert("RGBA")
    logo_path = prepared_logo_path if logo_source_mode == "uploaded_artwork" else None
    logo_guard = {"mode": logo_mode, "ok": None, "size_ratio": None}
    halftone_final_used = False

    if make_halftone_logo_qr and use_halftone_as_final and qr is not None and logo_path:
        best_halftone, halftone_rows = render_best_readable_halftone_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            logo_path=logo_path,
            expected_data=expected_data,
            out_dir="qr_fusion_outputs",
            prefix="halftone_logo_qr_preview",
        )
        if best_halftone:
            halftone_preview_path = "qr_fusion_outputs/halftone_logo_qr_preview.png"
            save_rgb_or_rgba(best_halftone["image"], halftone_preview_path)
            final_img = best_halftone["image"]
            logo_guard = {"mode": "logo_aware_halftone", "ok": True, "size_ratio": "module_texture"}
            halftone_final_used = True
        else:
            print("Halftone adaylar okunabilirlikten gecmedi; final icin duz logo guard uygulanacak. Okunmayan halftone teslim dosyasina alinmayacak.")

    if not halftone_final_used:
        final_img, logo_guard = apply_logo_treatment_readable(
            final_img,
            logo_path,
            qr=qr,
            border=quiet_zone,
            expected_data=expected_data,
            mode=logo_mode,
            size_ratio=logo_size_ratio,
            badge_shape=logo_badge_shape,
            badge_opacity=logo_badge_opacity,
            badge_padding=logo_badge_padding,
            visual_strength=logo_visual_strength,
            auto_guard=logo_auto_readability_guard,
            test_path="qr_fusion_outputs/_final_logo_probe.png",
        )

    save_rgb_or_rgba(final_img, final_path)
    final_result = test_qr_all(final_path, expected_data=expected_data)
    print("Final secildi:", final_path)
    print("Strength:", best_strength, "Score:", round(best_score, 2), "Decoder:", final_result["passed_engines"])
    print(f"Logo mode uygulandi: {logo_guard.get('mode', logo_mode)} | ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
    print("Okunan veri:", final_result["texts"])
    display(Image.open(final_path))


<!-- artqr-doc -->
## 10. Adaylari Karsilastirma

Hazirlanan gorsel, temiz QR ve secilen aday tek bir yan yana onizleme icinde gosterilir. Bu hucre, hangi aday dosyasinin hem guzel gorundugunu hem de okunabilir kaldigini hizli kontrol etmek icin kullanilir.


In [ ]:
#@title 10. Aday Inceleme Formu
inspect_candidate_path = "" #@param {type:"string"}

if not inspect_candidate_path.strip():
    if candidate_paths:
        inspect_candidate_path = candidate_paths[-1]
    else:
        raise ValueError("Aday bulunamadi. Once Hucre 7'yi calistir.")

prepared = Image.open("qr_fusion_outputs/prepared_artwork.png").convert("RGB").resize((384, 384), Image.Resampling.LANCZOS)
clean = Image.open("qr_fusion_outputs/clean_qr.png").convert("RGB").resize((384, 384), Image.Resampling.NEAREST)
candidate = Image.open(inspect_candidate_path).convert("RGB").resize((384, 384), Image.Resampling.LANCZOS)

canvas = Image.new("RGB", (384 * 3, 384), "white")
canvas.paste(prepared, (0, 0))
canvas.paste(clean, (384, 0))
canvas.paste(candidate, (384 * 2, 0))

print("Incelenen aday:", inspect_candidate_path)
print(test_qr_all(inspect_candidate_path, expected_data=expected_data))
display(canvas)


<!-- artqr-doc -->
## 11. Manuel Final Secimi

Otomatik secilen sonuc yerine belirli bir aday dosyasini final yapmak isterseniz dosya yolunu burada verebilirsiniz. Secilen dosya once QR testinden gecirilir; okunabiliyorsa final dosyasi olarak kaydedilir.


In [ ]:
#@title 11. Manuel Final Secimi Formu
manual_final_path = "" #@param {type:"string"}

if manual_final_path.strip():
    final_path = "qr_fusion_outputs/final_readable_art_qr.png"
    result = test_qr_all(manual_final_path, expected_data=expected_data)
    if not result["ok"]:
        raise ValueError(f"Secilen aday QR testten gecmedi: {result}")
    Image.open(manual_final_path).save(final_path, quality=95)
    print("Manuel final secildi:", final_path)
    print("Okunan veri:", result["texts"])
    display(Image.open(final_path))
else:
    print("Manuel secim yapilmadi. Hucre 7'nin sectigi final kullanilacak.")


<!-- artqr-doc -->
## 12. Ticari Teslim Paketi Uretme

Bu hucre soft, balanced ve strong gibi farkli tasarim siddetlerinde ticari varyasyonlar uretir. Logo, rozet, renkli modul, saydam PNG, yuksek cozunurluklu PNG/JPG ve paket ZIP dosyasi gibi teslim ciktilari burada hazirlanir.


In [ ]:
# Bu blok ticari teslim paketini uretir: logo-aware halftone final, alternatif QR stilleri, print dosyalari ve raporlar.
delivery_files = []
commercial_rows = []

if commercial_pack_enabled:
    os.makedirs("qr_fusion_outputs/commercial", exist_ok=True)
    logo_path = prepared_logo_path if logo_source_mode == "uploaded_artwork" else None
    enable_all_delivery_variants = globals().get("enable_all_delivery_variants", False)

    def remember_delivery(path, result=None, force=False):
        # Normal teslimde okunabilir dosyalar eklenir; full_output_explorer modunda
        # gorsel arama icin NO sonuc veren varyasyonlar da ZIP'e girer.
        if not path or not os.path.exists(path):
            return
        if result is None or result.get("ok") or force:
            if path not in delivery_files:
                delivery_files.append(path)
        elif result is not None:
            print("Okunmayan gorsel varyant sadece raporda birakildi:", path)

    if make_soft_balanced_strong:
        commercial_presets = [
            ("soft", max(dark_strength_start, 0.28), 1.05, 1.15),
            ("balanced", max((dark_strength_start + dark_strength_end) / 2, 0.45), contrast_after, sharpness_after),
            ("strong", min(dark_strength_end, 0.90), max(contrast_after, 1.18), max(sharpness_after, 1.45)),
        ]
        for name, strength, contrast_value, sharpness_value in commercial_presets:
            img = luminance_fusion(
                art_img=art_img,
                qr_img=qr_img,
                qr=qr,
                border=quiet_zone,
                dark_strength=float(strength),
                light_ratio=light_strength_ratio,
                finder_boost=finder_boost,
                quiet_boost=quiet_zone_boost,
                module_contrast=module_contrast,
                center_bias=center_bias,
            )
            img = polish_image(img, contrast=contrast_value, sharpness=sharpness_value)
            img, logo_guard = apply_logo_treatment_readable(
                img,
                logo_path,
                qr=qr,
                border=quiet_zone,
                expected_data=expected_data,
                mode=logo_mode,
                size_ratio=logo_size_ratio,
                badge_shape=logo_badge_shape,
                badge_opacity=logo_badge_opacity,
                badge_padding=logo_badge_padding,
                visual_strength=logo_visual_strength,
                auto_guard=logo_auto_readability_guard,
                test_path=f"qr_fusion_outputs/commercial/_logo_probe_{name}.png",
            )
            print(f"Logo guard {name}: ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
            path = f"qr_fusion_outputs/commercial/art_qr_{name}.png"
            save_rgb_or_rgba(img, path)
            result = test_qr_all(path, expected_data=expected_data)
            commercial_rows.append({"path": path, "result": result})
            print(path, "OK" if result["ok"] else "NO", result["texts"])
            remember_delivery(path, result, force=include_unreadable_visual_variants)

    if make_styled_clean_qr and qr is not None:
        styled = render_styled_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            module_shape=module_shape,
            finder_style=finder_style,
            color_mode=color_mode,
            background_style=background_style,
            transparent=False,
            artwork=art_img,
        )
        styled, logo_guard = apply_logo_treatment_readable(
            styled,
            logo_path,
            qr=qr,
            border=quiet_zone,
            expected_data=expected_data,
            mode=logo_mode,
            size_ratio=logo_size_ratio,
            badge_shape=logo_badge_shape,
            badge_opacity=logo_badge_opacity,
            badge_padding=logo_badge_padding,
            visual_strength=logo_visual_strength,
            auto_guard=logo_auto_readability_guard,
            test_path="qr_fusion_outputs/commercial/_logo_probe_styled.png",
        )
        print(f"Logo guard styled: ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
        styled_path = "qr_fusion_outputs/commercial/styled_clean_qr.png"
        save_rgb_or_rgba(styled, styled_path)
        result = test_qr_all(styled_path, expected_data=expected_data)
        commercial_rows.append({"path": styled_path, "result": result})
        print(styled_path, "OK" if result["ok"] else "NO", result["texts"])
        remember_delivery(styled_path, result, force=include_unreadable_visual_variants)

    if enable_all_delivery_variants and qr is not None:
        styled_variants = [
            dict(name="clean_rounded_brand", module_shape="rounded", finder_style="circle", color_mode="brand_color", background_style="white"),
            dict(name="clean_square_bw", module_shape="square", finder_style="classic", color_mode="black_white", background_style="white"),
            dict(name="clean_dot_brand", module_shape="dot", finder_style="circle", color_mode="brand_color", background_style="white"),
            dict(name="clean_diamond_gradient", module_shape="diamond", finder_style="circle", color_mode="gradient", background_style="white"),
            dict(name="clean_soft_blob_gradient", module_shape="soft_blob", finder_style="rounded", color_mode="gradient", background_style="white"),
            dict(name="clean_vertical_line_brand", module_shape="vertical_line", finder_style="rounded", color_mode="brand_color", background_style="white"),
            dict(name="clean_horizontal_line_brand", module_shape="horizontal_line", finder_style="rounded", color_mode="brand_color", background_style="white"),
            dict(name="qrbtf_interlock_line", module_shape="interlock_line", finder_style="planet", color_mode="brand_color", background_style="white"),
            dict(name="qrbtf_cross_line", module_shape="cross_line", finder_style="classic", color_mode="brand_color", background_style="white"),
            dict(name="qrbtf_radial_line", module_shape="radial_line", finder_style="circle", color_mode="gradient", background_style="white"),
            dict(name="qrbtf_diagonal_tl_br", module_shape="diagonal_tl_br", finder_style="rounded", color_mode="brand_color", background_style="white"),
            dict(name="qrbtf_diagonal_tr_bl", module_shape="diagonal_tr_bl", finder_style="rounded", color_mode="brand_color", background_style="white"),
            dict(name="clean_image_based_artwork", module_shape="rounded", finder_style="rounded", color_mode="image_based", background_style="artwork"),
        ]
        for spec in styled_variants:
            variant = render_styled_qr(
                qr=qr,
                qr_img=qr_img,
                size=output_size,
                border=quiet_zone,
                module_shape=spec["module_shape"],
                finder_style=spec["finder_style"],
                color_mode=spec["color_mode"],
                background_style=spec["background_style"],
                transparent=False,
                artwork=art_img,
            )
            if logo_path and logo_mode != "none":
                variant, logo_guard = apply_logo_treatment_readable(
                    variant,
                    logo_path,
                    qr=qr,
                    border=quiet_zone,
                    expected_data=expected_data,
                    mode=logo_mode,
                    size_ratio=logo_size_ratio,
                    badge_shape=logo_badge_shape,
                    badge_opacity=logo_badge_opacity,
                    badge_padding=logo_badge_padding,
                    visual_strength=logo_visual_strength,
                    auto_guard=logo_auto_readability_guard,
                    test_path=f"qr_fusion_outputs/commercial/_logo_probe_{spec['name']}.png",
                )
            variant_path = f"qr_fusion_outputs/commercial/variant_{spec['name']}.png"
            save_rgb_or_rgba(variant, variant_path)
            result = test_qr_all(variant_path, expected_data=expected_data)
            commercial_rows.append({"path": variant_path, "result": result, "variant": spec})
            print(variant_path, "OK" if result["ok"] else "NO", result["texts"])
            remember_delivery(variant_path, result, force=include_unreadable_visual_variants)

        if logo_path:
            logo_variants = ["center_badge", "center_transparent", "corner_small", "qr_over_logo_soft"]
            base_for_logo_variants = Image.open("qr_fusion_outputs/final_readable_art_qr.png").convert("RGBA") if os.path.exists("qr_fusion_outputs/final_readable_art_qr.png") else None
            if base_for_logo_variants is not None:
                for mode_name in logo_variants:
                    variant, logo_guard = apply_logo_treatment_readable(
                        base_for_logo_variants,
                        logo_path,
                        qr=qr,
                        border=quiet_zone,
                        expected_data=expected_data,
                        mode=mode_name,
                        size_ratio=logo_size_ratio,
                        badge_shape=logo_badge_shape,
                        badge_opacity=logo_badge_opacity,
                        badge_padding=logo_badge_padding,
                        visual_strength=logo_visual_strength,
                        auto_guard=logo_auto_readability_guard,
                        test_path=f"qr_fusion_outputs/commercial/_logo_probe_logo_{mode_name}.png",
                    )
                    variant_path = f"qr_fusion_outputs/commercial/logo_placement_{mode_name}.png"
                    save_rgb_or_rgba(variant, variant_path)
                    result = test_qr_all(variant_path, expected_data=expected_data)
                    commercial_rows.append({"path": variant_path, "result": result, "logo_mode": mode_name, "logo_guard": logo_guard})
                    print(variant_path, "OK" if result["ok"] else "NO", result["texts"], "guard", logo_guard)
                    remember_delivery(variant_path, result, force=include_unreadable_visual_variants)

    if make_halftone_logo_qr and qr is not None and logo_path:
        best_halftone, halftone_rows = render_best_readable_halftone_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            logo_path=logo_path,
            expected_data=expected_data,
            out_dir="qr_fusion_outputs/commercial",
            prefix="halftone_logo_qr",
        )
        for row in halftone_rows:
            commercial_rows.append({"path": row["path"], "result": row["result"], "visual": row["quality"].get("visual"), "quality": row["quality"], "variant": row["profile"]})
        if best_halftone:
            halftone_path = "qr_fusion_outputs/commercial/halftone_logo_qr.png"
            save_rgb_or_rgba(best_halftone["image"], halftone_path)
            result = test_qr_all(halftone_path, expected_data=expected_data)
            delivery_files.append(halftone_path)
            if use_halftone_as_final:
                final_path = "qr_fusion_outputs/final_readable_art_qr.png"
                final_halftone_img = best_halftone["image"]
                if globals().get("halftone_apply_logo_badge_to_final", False) and logo_mode in ("center_badge", "center_transparent", "corner_small"):
                    final_halftone_img, logo_guard = apply_logo_treatment_readable(
                        final_halftone_img,
                        logo_path,
                        qr=qr,
                        border=quiet_zone,
                        expected_data=expected_data,
                        mode=logo_mode,
                        size_ratio=logo_size_ratio,
                        badge_shape=logo_badge_shape,
                        badge_opacity=logo_badge_opacity,
                        badge_padding=logo_badge_padding,
                        visual_strength=logo_visual_strength,
                        auto_guard=logo_auto_readability_guard,
                        test_path="qr_fusion_outputs/commercial/_logo_probe_halftone_final_badge.png",
                    )
                    print(f"Halftone final logo badge uygulandi: mode={logo_guard.get('mode')} shape={logo_badge_shape} ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
                save_rgb_or_rgba(final_halftone_img, final_path)
                print("Logo-aware halftone QR final olarak secildi:", final_path)
        else:
            print("Halftone adaylar okuyucu testinden gecmedi; final icin okunabilir diger adaylar korunacak ve halftone teslim dosyasina alinmayacak.")

    if enable_all_delivery_variants and make_halftone_logo_qr and qr is not None and logo_path:
        halftone_variant_specs = [
            dict(name="halftone_plus_premium", mark_shape="plus", density=0.96, logo_strength=0.96, light_logo_alpha=0.94, outline_boost=0.72),
            dict(name="halftone_dot_soft", mark_shape="dot", density=0.90, logo_strength=0.90, light_logo_alpha=0.88, outline_boost=0.58),
            dict(name="halftone_square_bold", mark_shape="square", density=0.98, logo_strength=0.94, light_logo_alpha=0.90, outline_boost=0.66),
            dict(name="halftone_diamond_editorial", mark_shape="diamond", density=0.92, logo_strength=0.92, light_logo_alpha=0.88, outline_boost=0.62),
            dict(name="halftone_star_accent", mark_shape="star", density=0.88, logo_strength=0.88, light_logo_alpha=0.84, outline_boost=0.56),
        ]
        for spec in halftone_variant_specs:
            halftone_variant = render_halftone_logo_qr(
                qr=qr,
                qr_img=qr_img,
                size=output_size,
                border=quiet_zone,
                logo_path=logo_path,
                mark_shape=spec["mark_shape"],
                density=spec["density"],
                logo_strength=spec["logo_strength"],
                background_dots=halftone_background_dots,
                designer_finders=halftone_designer_finders,
                logo_forward=halftone_logo_forward,
                light_logo_alpha=spec["light_logo_alpha"],
                logo_cell_threshold=halftone_logo_cell_threshold,
                logo_aware=halftone_logo_aware_mode,
                outline_boost=spec["outline_boost"],
                edge_weight=halftone_edge_weight,
                negative_cutout=halftone_negative_cutout,
                safe_background_density=halftone_safe_background_density,
            )
            variant_path = f"qr_fusion_outputs/commercial/{spec['name']}.png"
            save_rgb_or_rgba(halftone_variant, variant_path)
            result = test_qr_all(variant_path, expected_data=expected_data)
            visual = visual_quality_metrics(variant_path, logo_path=logo_path)
            quality = commercial_quality_grade(variant_path, result=result, logo_path=logo_path)
            commercial_rows.append({"path": variant_path, "result": result, "visual": visual, "quality": quality, "variant": spec})
            print(variant_path, "OK" if result["ok"] else "NO", result["texts"], "visual", visual)
            remember_delivery(variant_path, result, force=include_unreadable_visual_variants)

    if make_transparent_png and qr is not None:
        transparent_qr = render_styled_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            module_shape=module_shape,
            finder_style=finder_style,
            color_mode=color_mode,
            background_style="transparent",
            transparent=True,
            artwork=None,
        )
        transparent_qr, logo_guard = apply_logo_treatment_readable(
            transparent_qr,
            logo_path,
            qr=qr,
            border=quiet_zone,
            expected_data=expected_data,
            mode=logo_mode,
            size_ratio=logo_size_ratio,
            badge_shape=logo_badge_shape,
            badge_opacity=logo_badge_opacity,
            badge_padding=logo_badge_padding,
            visual_strength=logo_visual_strength,
            auto_guard=logo_auto_readability_guard,
            test_path="qr_fusion_outputs/commercial/_logo_probe_transparent.png",
        )
        print(f"Logo guard transparent: ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
        transparent_path = "qr_fusion_outputs/commercial/transparent_logo_qr.png"
        save_rgb_or_rgba(transparent_qr, transparent_path)
        # QR okuyucular transparan zemini farkli yorumlayabilir; test icin beyaz zemine kompozitlenir.
        test_canvas = Image.new("RGBA", transparent_qr.size, (255, 255, 255, 255))
        test_canvas.alpha_composite(transparent_qr)
        transparent_test_path = "qr_fusion_outputs/commercial/transparent_logo_qr_white_test.png"
        save_rgb_or_rgba(test_canvas, transparent_test_path)
        result = test_qr_all(transparent_test_path, expected_data=expected_data)
        commercial_rows.append({"path": transparent_path, "result": result})
        print(transparent_path, "OK" if result["ok"] else "NO", result["texts"])
        remember_delivery(transparent_path, result, force=include_unreadable_visual_variants)
        remember_delivery(transparent_test_path, result, force=include_unreadable_visual_variants)

    if (make_print_png or make_print_pdf) and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        print_img = Image.open("qr_fusion_outputs/final_readable_art_qr.png").convert("RGB")
        print_w, print_h, active_dpi, print_in_w, print_in_h = print_dimensions_px()
        print_img = print_img.resize((print_w, print_h), Image.Resampling.LANCZOS)

        if make_print_png:
            print_path = f"qr_fusion_outputs/commercial/print_ready_{print_in_w:g}x{print_in_h:g}in_{active_dpi}dpi.png"
            print_img.save(print_path, quality=95, dpi=(active_dpi, active_dpi))
            result = test_qr_all(print_path, expected_data=expected_data)
            commercial_rows.append({"path": print_path, "result": result})
            print(print_path, "OK" if result["ok"] else "NO", result["texts"])
            remember_delivery(print_path, result, force=include_unreadable_visual_variants)

        if make_print_pdf:
            pdf_path = f"qr_fusion_outputs/commercial/print_ready_{print_in_w:g}x{print_in_h:g}in_{active_dpi}dpi.pdf"
            print_img.save(pdf_path, "PDF", resolution=float(active_dpi))
            delivery_files.append(pdf_path)
            print("Baski PDF hazir:", pdf_path)

    if make_scan_me_frame and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        framed_path = "qr_fusion_outputs/commercial/final_scan_me_frame.png"
        add_scan_me_frame(Image.open("qr_fusion_outputs/final_readable_art_qr.png"), framed_path, text=scan_me_text, style=scan_frame_style)
        result = test_qr_all(framed_path, expected_data=expected_data)
        commercial_rows.append({"path": framed_path, "result": result, "frame_style": scan_frame_style})
        print(framed_path, "OK" if result["ok"] else "NO", result["texts"])
        remember_delivery(framed_path, result, force=include_unreadable_visual_variants)

    if make_social_preview:
        # Sosyal onizleme final dosyayi tercih eder; boylece halftone final secildiyse pazarlama gorseli de onu kullanir.
        source_for_social = "qr_fusion_outputs/final_readable_art_qr.png"
        if not os.path.exists(source_for_social) and delivery_files:
            source_for_social = delivery_files[0]
        if os.path.exists(source_for_social):
            social_path = "qr_fusion_outputs/commercial/social_preview_1080.png"
            create_social_preview(Image.open(source_for_social), social_path)
            delivery_files.append(social_path)
            print("Sosyal medya onizleme hazir:", social_path)

    if make_styled_clean_qr and qr is None:
        print("Styled clean QR, hazir raster QR modunda atlandi.")

    if make_transparent_png and qr is None:
        print("Transparent styled QR, hazir raster QR modunda atlandi.")

    if make_jpg and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        jpg_path = "qr_fusion_outputs/commercial/final_readable_art_qr.jpg"
        save_rgb_or_rgba(Image.open("qr_fusion_outputs/final_readable_art_qr.png"), jpg_path)
        result = test_qr_all(jpg_path, expected_data=expected_data)
        commercial_rows.append({"path": jpg_path, "result": result})
        remember_delivery(jpg_path, result, force=include_unreadable_visual_variants)
        print(jpg_path, "OK" if result["ok"] else "NO", result["texts"])

    if make_svg or make_eps:
        vector_paths = save_segno_vector(
            qr,
            "qr_fusion_outputs/commercial/clean_qr_vector.svg",
            "qr_fusion_outputs/commercial/clean_qr_vector.eps" if make_eps else None,
        )
        delivery_files.extend(vector_paths)
        if vector_paths:
            print("Vector temiz QR hazir:", vector_paths)
        elif upload_mode == "use_uploaded_qr_image":
            print("SVG/EPS sadece Segno ile uretilen QR icin hazirlanir; hazir raster QR icin atlandi.")

    if make_layout_pack and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        layout_source = Image.open("qr_fusion_outputs/final_readable_art_qr.png")
        for layout in ["sticker", "business_card", "label"]:
            layout_path = f"qr_fusion_outputs/commercial/{layout}_layout.png"
            create_layout_canvas(layout_source, layout_path, layout=layout, title=brand_text)
            delivery_files.append(layout_path)
            print("Layout hazir:", layout_path)

    shape_quality_diagnostics = build_shape_quality_diagnostics(commercial_rows, logo_path=logo_path)
    if shape_quality_diagnostics:
        shape_report_path = "qr_fusion_outputs/commercial/shape_quality_diagnosis.txt"
        write_shape_quality_report(shape_report_path, shape_quality_diagnostics)
        delivery_files.append(shape_report_path)
        print("Sekil bazli kalite tespiti hazir:", shape_report_path)
        for item in shape_quality_diagnostics[:5]:
            print("Sekil kalite:", item["shape"], item["grade"], item["score"], item["path"])

    if make_test_report:
        report_path = "qr_fusion_outputs/commercial/qr_test_report.txt"
        write_test_report(report_path, commercial_rows)
        delivery_files.append(report_path)
        print("Test raporu hazir:", report_path)

    if os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        final_delivery_result = test_qr_all("qr_fusion_outputs/final_readable_art_qr.png", expected_data=expected_data)
        brief_path = "qr_fusion_outputs/commercial/customer_delivery_brief.txt"
        write_customer_delivery_brief(brief_path, final_delivery_result, delivery_files)
        delivery_files.append(brief_path)
        checklist_path = "qr_fusion_outputs/commercial/gig_readiness_checklist.txt"
        write_gig_readiness_checklist(checklist_path, final_delivery_result, delivery_files)
        delivery_files.append(checklist_path)
        quality_summary_path = "qr_fusion_outputs/commercial/client_quality_summary.txt"
        write_client_quality_summary(quality_summary_path, "qr_fusion_outputs/final_readable_art_qr.png", final_delivery_result, globals().get("shape_quality_diagnostics", []))
        delivery_files.append(quality_summary_path)
        print("Musteri teslim brief'i hazir:", brief_path)
        print("Gig hazirlik checklist'i hazir:", checklist_path)
        print("Musteri kalite ozeti hazir:", quality_summary_path)

    if enable_all_delivery_variants:
        variant_index_path = "qr_fusion_outputs/commercial/variant_index.json"
        with open(variant_index_path, "w", encoding="utf-8") as f:
            json.dump(commercial_rows, f, ensure_ascii=False, indent=2, default=str)
        delivery_files.append(variant_index_path)
        print("Varyasyon indeksi hazir:", variant_index_path)

    print("Teslim dosyalari:")
    for path in delivery_files:
        print("-", path)
else:
    delivery_files = []
    print("Ticari paket kapali. Sadece klasik final uretilecek.")




<!-- artqr-doc -->
## 13. Final Dosyayi Bulma, Paketleme ve Indirme

Final QR dosyasi yoksa okunabilir bir yedek dosya aranir ve final olarak kaydedilir. Son olarak cikti klasoru ZIP haline getirilir, dosyalar listelenir ve Colab uzerinden indirme baslatilir.


In [ ]:
# Bu blok son guvenlik kontroludur: final QR okunabilir mi test eder, teslim zip dosyasini olusturur ve indirmeye hazirlar.
final_path = "qr_fusion_outputs/final_readable_art_qr.png"

def find_readable_fallback():
    fallback_paths = []

    for item in globals().get("passed_candidates", []):
        if isinstance(item, (tuple, list)) and item:
            fallback_paths.append(item[0])

    fallback_paths.extend(globals().get("delivery_files", []))
    fallback_paths.extend(globals().get("candidate_paths", []))
    fallback_paths.append("qr_fusion_outputs/halftone_logo_qr_preview.png")

    commercial_dir = "qr_fusion_outputs/commercial"
    if os.path.isdir(commercial_dir):
        for name in sorted(os.listdir(commercial_dir)):
            if name.lower().endswith((".png", ".jpg", ".jpeg")):
                fallback_paths.append(os.path.join(commercial_dir, name))

    seen = set()
    for path in fallback_paths:
        if not path or path in seen or not os.path.exists(path):
            continue
        seen.add(path)
        try:
            result = test_qr_all(path, expected_data=expected_data)
        except Exception:
            continue
        if result["ok"]:
            return path, result

    return None, None

if not os.path.exists(final_path):
    fallback_path, fallback_result = find_readable_fallback()
    if fallback_path:
        Image.open(fallback_path).save(final_path, quality=95)
        print("Final dosya yoktu; okunabilir adaydan olusturuldu:", fallback_path)
        print("Okunan veri:", fallback_result["texts"])
    else:
        raise FileNotFoundError(
            "Final dosya bulunamadi ve okunabilir aday da bulunamadi. "
            "Once aday uretim hucrelerini calistir. Adaylar NO cikiyorsa "
            "dark_strength_end=0.95, module_contrast=0.85, finder_boost=2.50, "
            "quiet_zone_boost=1.00 yapip tekrar dene."
        )

result = test_qr_all(final_path, expected_data=expected_data)
print("Final QR test:", result)

if not result["ok"]:
    raise ValueError("Final QR okunamadi. Ayarlari artirip tekrar dene.")

zip_path = "qr_fusion_delivery.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(final_path, arcname="final_readable_art_qr.png")
    z.write("qr_fusion_outputs/clean_qr.png", arcname="clean_qr.png")
    z.write("qr_fusion_outputs/prepared_artwork.png", arcname="prepared_artwork.png")
    if os.path.exists("qr_fusion_outputs/prepared_logo_fullscreen.png"):
        z.write("qr_fusion_outputs/prepared_logo_fullscreen.png", arcname="prepared_logo_fullscreen.png")

    extra_delivery_paths = list(globals().get("delivery_files", []))
    if os.path.exists("qr_fusion_outputs/halftone_logo_qr_preview.png"):
        extra_delivery_paths.append("qr_fusion_outputs/halftone_logo_qr_preview.png")

    seen_extra_paths = set()
    for extra_path in extra_delivery_paths:
        if os.path.exists(extra_path) and extra_path not in seen_extra_paths:
            seen_extra_paths.add(extra_path)
            z.write(extra_path, arcname=os.path.relpath(extra_path, "qr_fusion_outputs"))

    manifest_path = "qr_fusion_outputs/delivery_manifest.txt"
    from datetime import datetime
    with open(manifest_path, "w", encoding="utf-8") as manifest:
        manifest.write("ArtQR Fusion Delivery Manifest\n")
        manifest.write("==============================\n\n")
        manifest.write(f"created_at: {datetime.now().isoformat(timespec='seconds')}\n")
        manifest.write(f"master_style_choice: {globals().get('master_style_choice', '')}\n")
        manifest.write(f"service_tier: {globals().get('service_tier', '')}\n")
        manifest.write(f"portfolio_example: {globals().get('portfolio_example', '')}\n")
        manifest.write(f"usage_place: {globals().get('usage_place', '')}\n")
        manifest.write(f"qr_content_type: {globals().get('qr_content_type', '')}\n")
        manifest.write(f"qr_static_or_dynamic: {globals().get('qr_static_or_dynamic', '')}\n")
        manifest.write(f"print_size_preset: {globals().get('print_size_preset', '')}\n")
        manifest.write(f"print_dpi: {globals().get('print_dpi', '')}\n")
        manifest.write(f"final_ok: {result.get('ok')}\n")
        manifest.write(f"decoder_pass_count: {result.get('decoder_pass_count')}\n")
        final_quality = commercial_quality_grade(final_path, result=result, logo_path=globals().get("prepared_logo_path"))
        manifest.write(f"commercial_quality_grade: {final_quality.get('grade')}\n")
        manifest.write(f"commercial_quality_score: {final_quality.get('score')} / 100\n")
        manifest.write(f"passed_engines: {result.get('passed_engines')}\n")
        manifest.write(f"decoded_texts: {result.get('texts')}\n")
        manifest.write("vector_note: artistic/logo-fused QR is high-resolution raster; clean SVG/EPS is separate when included.\n")
        manifest.write("dynamic_note: this notebook creates static QR payloads unless the client provides a dynamic redirect URL.\n")
        manifest.write("print_note: test one physical sample before mass production.\n\n")
        manifest.write("zip_files:\n")
        manifest.write("- final_readable_art_qr.png\n")
        manifest.write("- clean_qr.png\n")
        manifest.write("- prepared_artwork.png\n")
        if os.path.exists("qr_fusion_outputs/prepared_logo_fullscreen.png"):
            manifest.write("- prepared_logo_fullscreen.png\n")
        for extra_path in extra_delivery_paths:
            if os.path.exists(extra_path):
                manifest.write(f"- {os.path.relpath(extra_path, 'qr_fusion_outputs')}\n")
    z.write(manifest_path, arcname="delivery_manifest.txt")



def display_final_delivery_gallery(max_items=12, thumb_size=360):
    gallery_paths = [final_path, "qr_fusion_outputs/clean_qr.png"]
    gallery_paths.extend(globals().get("delivery_files", []))
    commercial_dir = "qr_fusion_outputs/commercial"
    if os.path.isdir(commercial_dir):
        for name in sorted(os.listdir(commercial_dir)):
            if name.lower().endswith((".png", ".jpg", ".jpeg")):
                gallery_paths.append(os.path.join(commercial_dir, name))

    seen = set()
    thumbs = []
    for path in gallery_paths:
        if not path or path in seen or not os.path.exists(path):
            continue
        seen.add(path)
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            continue
        img.thumbnail((thumb_size, thumb_size), Image.Resampling.LANCZOS)
        tile = Image.new("RGB", (thumb_size, thumb_size + 54), "white")
        tile.paste(img, ((thumb_size - img.size[0]) // 2, 0))
        draw = ImageDraw.Draw(tile)
        label = os.path.basename(path)[:42]
        draw.text((10, thumb_size + 10), label, fill=(20, 20, 20))
        thumbs.append(tile)
        if len(thumbs) >= max_items:
            break

    if not thumbs:
        return
    cols = min(3, len(thumbs))
    rows = int(math.ceil(len(thumbs) / cols))
    canvas = Image.new("RGB", (cols * thumb_size, rows * (thumb_size + 54)), "white")
    for idx, tile in enumerate(thumbs):
        canvas.paste(tile, ((idx % cols) * thumb_size, (idx // cols) * (thumb_size + 54)))
    print("Final onizleme galerisi: final, temiz QR ve teslim varyasyonlari")
    display(canvas)

display_final_delivery_gallery()

print("Teslim paketi hazir:", zip_path)

if download_final and files is not None:
    files.download(zip_path)
elif download_final:
    print("Colab download araci yok; ZIP dosyasi bu yolda hazir:", zip_path)

